Set up directories and generate all SIMULATED datasets

In [1]:

import os
import numpy as np
import pandas as pd

np.random.seed(42)

# Create output directories
os.makedirs('/mnt/results/data', exist_ok=True)
os.makedirs('/mnt/results/figures', exist_ok=True)

# ─────────────────────────────────────────────
# DATASET 1: TCGA LUAD TP53 mutation landscape
# 295 TP53-mutant samples
# ─────────────────────────────────────────────
n_mut = 295
n_wt  = 271

mut_types = ['Missense', 'Nonsense', 'Frameshift', 'Splice Site', 'In-frame Deletion']
# Realistic TP53 mutation type distribution in LUAD
mut_probs = [0.62, 0.18, 0.10, 0.07, 0.03]

mut_samples = [f'TCGA-LUAD-MUT-{i:04d}' for i in range(1, n_mut+1)]
wt_samples  = [f'TCGA-LUAD-WT-{i:04d}'  for i in range(1, n_wt+1)]

mut_type_list = np.random.choice(mut_types, size=n_mut, p=mut_probs)

# Codon positions (hotspots: R175, G245, R248, R249, R273, R282)
hotspot_codons = [175, 245, 248, 249, 273, 282]
other_codons   = list(range(100, 393))
codon_choices  = hotspot_codons * 8 + other_codons  # weight hotspots
codon_list     = np.random.choice(codon_choices, size=n_mut)

# VAF (variant allele frequency)
vaf_list = np.clip(np.random.beta(3, 2, size=n_mut), 0.05, 0.99)

# TMB (tumor mutation burden) — higher in TP53-mut
tmb_mut = np.random.lognormal(mean=2.8, sigma=0.6, size=n_mut)
tmb_wt  = np.random.lognormal(mean=2.2, sigma=0.5, size=n_wt)

df_mut = pd.DataFrame({
    'sample_id':    mut_samples,
    'tp53_status':  'Mutant',
    'mutation_type': mut_type_list,
    'codon_position': codon_list,
    'vaf':           np.round(vaf_list, 3),
    'tmb':           np.round(tmb_mut, 2),
    'stage':         np.random.choice(['I','II','III','IV'], size=n_mut, p=[0.35,0.25,0.25,0.15]),
    'smoking_status': np.random.choice(['Smoker','Former','Never'], size=n_mut, p=[0.45,0.40,0.15]),
})

df_wt = pd.DataFrame({
    'sample_id':    wt_samples,
    'tp53_status':  'WT',
    'mutation_type': 'WT',
    'codon_position': np.nan,
    'vaf':           np.nan,
    'tmb':           np.round(tmb_wt, 2),
    'stage':         np.random.choice(['I','II','III','IV'], size=n_wt, p=[0.38,0.27,0.22,0.13]),
    'smoking_status': np.random.choice(['Smoker','Former','Never'], size=n_wt, p=[0.40,0.38,0.22]),
})

df_tcga = pd.concat([df_mut, df_wt], ignore_index=True)
df_tcga.to_csv('/mnt/results/data/SIMULATED_TCGA_LUAD_TP53_mutations.csv', index=False)
print(f"TCGA dataset: {df_tcga.shape} | Mut: {n_mut} | WT: {n_wt}")
print(df_tcga['mutation_type'].value_counts())

# ─────────────────────────────────────────────
# DATASET 2: DepMap 93 SL candidates
# ─────────────────────────────────────────────
# Top 5 validated candidates (anchored to provided values)
top5 = {
    'gene':       ['SPC24', 'BUB1B', 'CDC45', 'PLK1', 'CDK1'],
    'dCERES':     [-0.175, -0.119, -0.144, -0.239, -0.201],
    'log2FC':     [1.13,    1.12,   1.26,   1.03,   1.00],
    'drug_status':['Novel', 'Novel','Novel','Clinical','Clinical'],
    'depmap_auc': [0.74,    0.71,   0.76,   0.82,   0.80],
}

# Generate remaining 88 candidates
n_rest = 88
gene_pool = [
    'AURKB','CENPE','KIF11','BIRC5','TTK','CHEK1','WEE1','PKMYT1',
    'CDC20','MAD2L1','BUB3','ZWINT','NDC80','NUF2','SPC25','MIS12',
    'CENPH','CENPN','INCENP','CDCA8','HASPIN','SGOL1','SGOL2','ESPL1',
    'PTTG1','CCNB1','CCNB2','CDC25C','CHEK2','ATR','ATM','BRCA1',
    'BRCA2','RAD51','FANCD2','FANCL','PALB2','BARD1','RPA1','RPA2',
    'PCNA','RFC1','RFC2','RFC3','RFC4','RFC5','LIG1','FEN1',
    'POLD1','POLD2','POLE','POLE2','MCM2','MCM3','MCM4','MCM5',
    'MCM6','MCM7','ORC1','ORC2','ORC3','ORC4','ORC5','ORC6',
    'CDT1','CDC6','GINS1','GINS2','GINS3','GINS4','TOPBP1','CLASPIN',
    'TIPIN','TIMELESS','AND1','RECQL4','HELQ','FANCM','FANCA','FANCB',
    'FANCC','FANCE','FANCF','FANCG','FANCI','FANCJ','FANCN',
]
gene_pool = gene_pool[:n_rest]

dCERES_rest = np.random.uniform(-0.22, 0.05, size=n_rest)
log2FC_rest = np.random.normal(0.5, 0.4, size=n_rest)
drug_rest   = np.random.choice(['Novel','Clinical','Preclinical'], size=n_rest, p=[0.55,0.25,0.20])
auc_rest    = np.random.uniform(0.45, 0.85, size=n_rest)
pval_rest   = np.random.uniform(0.001, 0.15, size=n_rest)
padj_rest   = np.minimum(pval_rest * n_rest, 1.0)

df_rest = pd.DataFrame({
    'gene':        gene_pool,
    'dCERES':      np.round(dCERES_rest, 3),
    'log2FC':      np.round(log2FC_rest, 3),
    'drug_status': drug_rest,
    'depmap_auc':  np.round(auc_rest, 3),
    'pvalue':      np.round(pval_rest, 4),
    'padj':        np.round(padj_rest, 4),
})

# Top 5 with anchored p-values
df_top5 = pd.DataFrame({
    'gene':        top5['gene'],
    'dCERES':      top5['dCERES'],
    'log2FC':      top5['log2FC'],
    'drug_status': top5['drug_status'],
    'depmap_auc':  top5['depmap_auc'],
    'pvalue':      [0.0008, 0.0021, 0.0012, 0.0001, 0.0003],
    'padj':        [0.0074, 0.0195, 0.0111, 0.0009, 0.0028],
})

df_depmap = pd.concat([df_top5, df_rest], ignore_index=True)
df_depmap.to_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv', index=False)
print(f"\nDepMap dataset: {df_depmap.shape}")
print(df_depmap.head(8))

# ─────────────────────────────────────────────
# DATASET 3: Summary table — top 10 candidates
# ─────────────────────────────────────────────
# Sort by combined score: most negative dCERES + highest log2FC
df_depmap['sl_score'] = -df_depmap['dCERES'] + df_depmap['log2FC']
top10 = df_depmap.nlargest(10, 'sl_score').reset_index(drop=True)
top10.to_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv', index=False)
print(f"\nTop 10 candidates:")
print(top10[['gene','dCERES','log2FC','drug_status','depmap_auc','padj']].to_string())


TCGA dataset: (566, 8) | Mut: 295 | WT: 271
mutation_type
WT                   271
Missense             180
Nonsense              52
Frameshift            34
Splice Site           24
In-frame Deletion      5
Name: count, dtype: int64
Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 104, in <module>
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/frame.py", line 778, in __init__
    mgr = dict_to_mgr(data, index, columns, dtype=dtype, copy=copy, typ=manager)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/internals/construction.py", line 503, in dict_to_mgr
    return arrays_to_mgr(arrays, columns, index, dtype=dtype, typ=typ, consolidate=copy)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/internals/construction.py", line 114, in arr

Generate all SIMULATED datasets with correct dimensions

In [3]:

import os
import numpy as np
import pandas as pd

np.random.seed(42)

os.makedirs('/mnt/results/data', exist_ok=True)
os.makedirs('/mnt/results/figures', exist_ok=True)

# ─────────────────────────────────────────────
# DATASET 1: TCGA LUAD TP53 mutation landscape
# ─────────────────────────────────────────────
n_mut, n_wt = 295, 271
mut_types = ['Missense', 'Nonsense', 'Frameshift', 'Splice Site', 'In-frame Deletion']
mut_probs  = [0.62, 0.18, 0.10, 0.07, 0.03]

mut_samples = [f'TCGA-LUAD-MUT-{i:04d}' for i in range(1, n_mut+1)]
wt_samples  = [f'TCGA-LUAD-WT-{i:04d}'  for i in range(1, n_wt+1)]

mut_type_list = np.random.choice(mut_types, size=n_mut, p=mut_probs)
hotspot_codons = [175, 245, 248, 249, 273, 282]
other_codons   = list(range(100, 393))
codon_choices  = hotspot_codons * 8 + other_codons
codon_list     = np.random.choice(codon_choices, size=n_mut)
vaf_list       = np.clip(np.random.beta(3, 2, size=n_mut), 0.05, 0.99)
tmb_mut        = np.random.lognormal(mean=2.8, sigma=0.6, size=n_mut)
tmb_wt         = np.random.lognormal(mean=2.2, sigma=0.5, size=n_wt)

df_mut = pd.DataFrame({
    'sample_id':      mut_samples,
    'tp53_status':    'Mutant',
    'mutation_type':  mut_type_list,
    'codon_position': codon_list,
    'vaf':            np.round(vaf_list, 3),
    'tmb':            np.round(tmb_mut, 2),
    'stage':          np.random.choice(['I','II','III','IV'], size=n_mut, p=[0.35,0.25,0.25,0.15]),
    'smoking_status': np.random.choice(['Smoker','Former','Never'], size=n_mut, p=[0.45,0.40,0.15]),
})
df_wt = pd.DataFrame({
    'sample_id':      wt_samples,
    'tp53_status':    'WT',
    'mutation_type':  'WT',
    'codon_position': np.nan,
    'vaf':            np.nan,
    'tmb':            np.round(tmb_wt, 2),
    'stage':          np.random.choice(['I','II','III','IV'], size=n_wt, p=[0.38,0.27,0.22,0.13]),
    'smoking_status': np.random.choice(['Smoker','Former','Never'], size=n_wt, p=[0.40,0.38,0.22]),
})
df_tcga = pd.concat([df_mut, df_wt], ignore_index=True)
df_tcga.to_csv('/mnt/results/data/SIMULATED_TCGA_LUAD_TP53_mutations.csv', index=False)
print(f"TCGA: {df_tcga.shape} | Mut={n_mut} | WT={n_wt}")
print(df_tcga['mutation_type'].value_counts())

# ─────────────────────────────────────────────
# DATASET 2: DepMap 93 SL candidates
# ─────────────────────────────────────────────
top5_data = {
    'gene':        ['SPC24', 'BUB1B', 'CDC45', 'PLK1', 'CDK1'],
    'dCERES':      [-0.175, -0.119, -0.144, -0.239, -0.201],
    'log2FC':      [1.13,    1.12,   1.26,   1.03,   1.00],
    'drug_status': ['Novel','Novel','Novel','Clinical','Clinical'],
    'depmap_auc':  [0.74,   0.71,   0.76,   0.82,   0.80],
    'pvalue':      [0.0008, 0.0021, 0.0012, 0.0001, 0.0003],
    'padj':        [0.0074, 0.0195, 0.0111, 0.0009, 0.0028],
}
df_top5 = pd.DataFrame(top5_data)

# Exactly 88 additional genes
gene_pool_88 = [
    'AURKB','CENPE','KIF11','BIRC5','TTK','CHEK1','WEE1','PKMYT1',
    'CDC20','MAD2L1','BUB3','ZWINT','NDC80','NUF2','SPC25','MIS12',
    'CENPH','CENPN','INCENP','CDCA8','HASPIN','SGOL1','SGOL2','ESPL1',
    'PTTG1','CCNB1','CCNB2','CDC25C','CHEK2','ATR','ATM','BRCA1',
    'BRCA2','RAD51','FANCD2','FANCL','PALB2','BARD1','RPA1','RPA2',
    'PCNA','RFC1','RFC2','RFC3','RFC4','RFC5','LIG1','FEN1',
    'POLD1','POLD2','POLE','POLE2','MCM2','MCM3','MCM4','MCM5',
    'MCM6','MCM7','ORC1','ORC2','ORC3','ORC4','ORC5','ORC6',
    'CDT1','CDC6','GINS1','GINS2','GINS3','GINS4','TOPBP1','CLASPIN',
    'TIPIN','TIMELESS','AND1','RECQL4','HELQ','FANCM','FANCA','FANCB',
    'FANCC','FANCE','FANCF','FANCG','FANCI','FANCJ','FANCN','FANCO',
]
assert len(gene_pool_88) == 88, f"Expected 88, got {len(gene_pool_88)}"

n_rest = 88
dCERES_rest = np.random.uniform(-0.22, 0.05, size=n_rest)
log2FC_rest = np.random.normal(0.5, 0.4, size=n_rest)
drug_rest   = np.random.choice(['Novel','Clinical','Preclinical'], size=n_rest, p=[0.55,0.25,0.20])
auc_rest    = np.random.uniform(0.45, 0.85, size=n_rest)
pval_rest   = np.random.uniform(0.001, 0.15, size=n_rest)
padj_rest   = np.minimum(pval_rest * 93, 1.0)

df_rest = pd.DataFrame({
    'gene':        gene_pool_88,
    'dCERES':      np.round(dCERES_rest, 3),
    'log2FC':      np.round(log2FC_rest, 3),
    'drug_status': drug_rest,
    'depmap_auc':  np.round(auc_rest, 3),
    'pvalue':      np.round(pval_rest, 4),
    'padj':        np.round(padj_rest, 4),
})

df_depmap = pd.concat([df_top5, df_rest], ignore_index=True)
assert len(df_depmap) == 93, f"Expected 93, got {len(df_depmap)}"
df_depmap.to_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv', index=False)
print(f"\nDepMap: {df_depmap.shape}")

# ─────────────────────────────────────────────
# DATASET 3: Top 10 candidates
# ─────────────────────────────────────────────
df_depmap['sl_score'] = -df_depmap['dCERES'] + df_depmap['log2FC']
top10 = df_depmap.nlargest(10, 'sl_score').reset_index(drop=True)
top10.to_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv', index=False)
print(f"\nTop 10 candidates:")
print(top10[['gene','dCERES','log2FC','drug_status','depmap_auc','padj']].to_string())


TCGA: (566, 8) | Mut=295 | WT=271
mutation_type
WT                   271
Missense             180
Nonsense              52
Frameshift            34
Splice Site           24
In-frame Deletion      5
Name: count, dtype: int64

DepMap: (93, 7)

Top 10 candidates:
    gene  dCERES  log2FC drug_status  depmap_auc    padj
0  CDC45  -0.144   1.260       Novel       0.760  0.0111
1  ESPL1  -0.153   1.235    Clinical       0.664  1.0000
2  POLD1  -0.208   1.157       Novel       0.490  1.0000
3  SPC24  -0.175   1.130       Novel       0.740  0.0074
4   PLK1  -0.239   1.030    Clinical       0.820  0.0009
5  BUB1B  -0.119   1.120       Novel       0.710  0.0195
6  CENPN  -0.141   1.098       Novel       0.766  0.4350
7   CDK1  -0.201   1.000    Clinical       0.800  0.0028
8  FANCN  -0.213   0.977       Novel       0.523  1.0000
9    ATM  -0.191   0.936       Novel       0.497  1.0000


Figure 1 — Waterfall plot: TP53 mutation landscape in TCGA LUAD

In [5]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

df_tcga = pd.read_csv('/mnt/results/data/SIMULATED_TCGA_LUAD_TP53_mutations.csv')
df_mut  = df_tcga[df_tcga['tp53_status'] == 'Mutant'].copy().reset_index(drop=True)

# Sort by VAF descending (waterfall style)
df_mut_sorted = df_mut.sort_values('vaf', ascending=False).reset_index(drop=True)

# ColorBrewer Set1-inspired, colorblind-friendly
color_map = {
    'Missense':          '#E41A1C',
    'Nonsense':          '#377EB8',
    'Frameshift':        '#4DAF4A',
    'Splice Site':       '#FF7F00',
    'In-frame Deletion': '#984EA3',
}

fig = plt.figure(figsize=(14, 8), facecolor='white')
gs  = GridSpec(3, 1, figure=fig, height_ratios=[3, 1, 1], hspace=0.08)

# ── Panel 1: VAF waterfall bars ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
x   = np.arange(len(df_mut_sorted))
bar_colors = [color_map[m] for m in df_mut_sorted['mutation_type']]
ax1.bar(x, df_mut_sorted['vaf'], color=bar_colors, width=1.0, linewidth=0)
ax1.set_xlim(-1, len(df_mut_sorted))
ax1.set_ylim(0, 1.05)
ax1.set_ylabel('Variant Allele Frequency (VAF)', fontsize=12)
ax1.set_title(
    'Figure 1 — TP53 Mutation Landscape in TCGA LUAD\n'
    '[SIMULATED DATA]  N = 295 TP53-mutant samples',
    fontsize=13, fontweight='bold', pad=10
)
ax1.axhline(0.5, color='#555555', lw=0.8, ls='--', alpha=0.6, label='VAF = 0.5')
ax1.set_xticks([])
ax1.spines[['top','right','bottom']].set_visible(False)

# Legend
patches = [mpatches.Patch(color=v, label=k) for k, v in color_map.items()]
ax1.legend(handles=patches, loc='upper right', fontsize=9,
           framealpha=0.9, title='Mutation Type', title_fontsize=9)

# ── Panel 2: Mutation type strip ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1])
for i, mtype in enumerate(df_mut_sorted['mutation_type']):
    ax2.bar(i, 1, color=color_map[mtype], width=1.0, linewidth=0)
ax2.set_xlim(-1, len(df_mut_sorted))
ax2.set_ylim(0, 1)
ax2.set_yticks([0.5])
ax2.set_yticklabels(['Mutation\nType'], fontsize=9)
ax2.set_xticks([])
ax2.spines[['top','right','bottom','left']].set_visible(False)

# ── Panel 3: TMB bar ──────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2])
ax3.bar(x, df_mut_sorted['tmb'], color='#1F78B4', width=1.0, linewidth=0, alpha=0.75)
ax3.set_xlim(-1, len(df_mut_sorted))
ax3.set_ylabel('TMB\n(mut/Mb)', fontsize=9)
ax3.set_xlabel(f'TP53-Mutant LUAD Samples (n = {len(df_mut_sorted)}, sorted by VAF)', fontsize=11)
ax3.spines[['top','right']].set_visible(False)
ax3.set_xticks([])

# Mutation type counts annotation
counts = df_mut_sorted['mutation_type'].value_counts()
count_text = '  '.join([f"{k}: {v}" for k, v in counts.items()])
fig.text(0.5, 0.01, count_text, ha='center', fontsize=9, color='#333333',
         style='italic')

plt.savefig('/mnt/results/figures/Figure1.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.savefig('/mnt/results/figures/Figure1.svg', bbox_inches='tight',
            facecolor='white')
plt.close()
print("Figure 1 saved.")


Figure 1 saved.


Figure 2 — Volcano plot: differential essentiality (DepMap)

In [7]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from adjustText import adjust_text

df = pd.read_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv')

# Per know-how guide: negative dCERES = more essential in TP53-mut
# X-axis: dCERES (negative = more essential), Y-axis: -log10(padj)
df['neg_log10_padj'] = -np.log10(df['padj'].clip(lower=1e-6))

# Significance thresholds
padj_thresh  = 0.05
dceres_thresh = -0.10   # meaningful differential essentiality

# Classify points
def classify(row):
    if row['padj'] <= padj_thresh and row['dCERES'] <= dceres_thresh:
        return 'Significant SL'
    elif row['padj'] <= padj_thresh:
        return 'Significant only'
    elif row['dCERES'] <= dceres_thresh:
        return 'dCERES only'
    else:
        return 'Not significant'

df['category'] = df.apply(classify, axis=1)

# ColorBrewer-inspired palette
cat_colors = {
    'Significant SL':   '#E41A1C',
    'Significant only': '#FF7F00',
    'dCERES only':      '#377EB8',
    'Not significant':  '#AAAAAA',
}
cat_sizes = {
    'Significant SL':   55,
    'Significant only': 40,
    'dCERES only':      40,
    'Not significant':  25,
}

top5 = ['SPC24', 'BUB1B', 'CDC45', 'PLK1', 'CDK1']

fig, ax = plt.subplots(figsize=(10, 7), facecolor='white')

for cat, grp in df.groupby('category'):
    ax.scatter(grp['dCERES'], grp['neg_log10_padj'],
               c=cat_colors[cat], s=cat_sizes[cat],
               alpha=0.80, linewidths=0.4, edgecolors='white',
               label=cat, zorder=3)

# Threshold lines
ax.axvline(dceres_thresh, color='#555555', lw=1.0, ls='--', alpha=0.7)
ax.axhline(-np.log10(padj_thresh), color='#555555', lw=1.0, ls='--', alpha=0.7)

# Annotate top 5
texts = []
for _, row in df[df['gene'].isin(top5)].iterrows():
    drug_label = '★' if row['drug_status'] == 'Clinical' else '◆'
    texts.append(
        ax.text(row['dCERES'], row['neg_log10_padj'],
                f" {drug_label} {row['gene']}",
                fontsize=9.5, fontweight='bold', color='#1A1A1A')
    )
adjust_text(texts, ax=ax,
            arrowprops=dict(arrowstyle='->', color='#444444', lw=0.8))

# Threshold annotations
ax.text(dceres_thresh + 0.003, ax.get_ylim()[1]*0.97,
        'dCERES = −0.10', fontsize=8, color='#555555', va='top')
ax.text(ax.get_xlim()[0] + 0.002, -np.log10(padj_thresh) + 0.05,
        'padj = 0.05', fontsize=8, color='#555555')

ax.set_xlabel('Differential CERES Score (TP53-mut − WT)\n← More essential in TP53-mut | Less essential →',
              fontsize=11)
ax.set_ylabel('−log₁₀(adjusted p-value)', fontsize=11)
ax.set_title('Figure 2 — Differential Essentiality Volcano Plot (DepMap)\n'
             '[SIMULATED DATA]  93 SL Candidates · TP53-Mutant vs WT LUAD Cell Lines',
             fontsize=12, fontweight='bold')

legend = ax.legend(title='Category', fontsize=9, title_fontsize=9,
                   framealpha=0.9, loc='upper right')

# Drug status legend
from matplotlib.lines import Line2D
extra = [Line2D([0],[0], marker='$★$', color='w', markerfacecolor='#333',
                markersize=10, label='Clinical drug exists'),
         Line2D([0],[0], marker='$◆$', color='w', markerfacecolor='#333',
                markersize=10, label='Novel (no drug)')]
ax.add_artist(legend)
ax.legend(handles=extra, loc='lower left', fontsize=9, framealpha=0.9,
          title='Drug Status', title_fontsize=9)

ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('white')

plt.tight_layout()
plt.savefig('/mnt/results/figures/Figure2.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/figures/Figure2.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure 2 saved.")
print(df['category'].value_counts())


Figure 2 saved.
category
Not significant    45
dCERES only        43
Significant SL      5
Name: count, dtype: int64


Figure 3 — Scatter: inverted essentiality score vs log2FC expression

In [9]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from adjustText import adjust_text

df = pd.read_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv')

# Per know-how guide: INVERT dCERES before correlating
# Inverted: positive = more essential
df['essentiality_inverted'] = -df['dCERES']

# Spearman correlation (inverted scores vs expression)
rho, pval = spearmanr(df['log2FC'], df['essentiality_inverted'])

# Color by drug status
drug_palette = {
    'Novel':       '#E41A1C',
    'Clinical':    '#377EB8',
    'Preclinical': '#FF7F00',
}
drug_sizes = {'Novel': 60, 'Clinical': 70, 'Preclinical': 50}

top5 = ['SPC24', 'BUB1B', 'CDC45', 'PLK1', 'CDK1']

fig, ax = plt.subplots(figsize=(10, 7), facecolor='white')

for drug, grp in df.groupby('drug_status'):
    ax.scatter(grp['log2FC'], grp['essentiality_inverted'],
               c=drug_palette.get(drug, '#AAAAAA'),
               s=drug_sizes.get(drug, 40),
               alpha=0.78, linewidths=0.4, edgecolors='white',
               label=drug, zorder=3)

# Reference lines
ax.axhline(0, color='#888888', lw=0.8, ls='--', alpha=0.6)
ax.axvline(1.0, color='#888888', lw=0.8, ls='--', alpha=0.6)

# Quadrant shading — top-right = high expression + high essentiality (SL zone)
xlim = ax.get_xlim()
ylim = ax.get_ylim()
ax.fill_between([1.0, max(df['log2FC'])+0.3],
                [0, 0], [max(df['essentiality_inverted'])+0.05]*2,
                color='#FFEDA0', alpha=0.35, zorder=1, label='SL Zone')

# Annotate top 5
texts = []
for _, row in df[df['gene'].isin(top5)].iterrows():
    texts.append(
        ax.text(row['log2FC'], row['essentiality_inverted'],
                f"  {row['gene']}", fontsize=9.5, fontweight='bold',
                color='#1A1A1A')
    )
adjust_text(texts, ax=ax,
            arrowprops=dict(arrowstyle='->', color='#444444', lw=0.8))

# Spearman annotation
ax.text(0.03, 0.97,
        f'Spearman ρ = {rho:.3f}\np = {pval:.4f}',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor='#CCCCCC', alpha=0.9))

ax.set_xlabel('log₂ Fold Change (TP53-mut / WT expression)', fontsize=11)
ax.set_ylabel('Essentiality Score (Inverted CERES)\n← Less essential | More essential →',
              fontsize=11)
ax.set_title('Figure 3 — Essentiality Score vs Expression in TP53-Mutant LUAD\n'
             '[SIMULATED DATA]  93 SL Candidates · DepMap CERES (inverted) vs TCGA log₂FC',
             fontsize=12, fontweight='bold')

ax.legend(title='Drug Status', fontsize=9, title_fontsize=9,
          framealpha=0.9, loc='lower right')
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('white')

plt.tight_layout()
plt.savefig('/mnt/results/figures/Figure3.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/figures/Figure3.svg', bbox_inches='tight', facecolor='white')
plt.close()
print(f"Figure 3 saved. Spearman ρ = {rho:.3f}, p = {pval:.4f}")


Figure 3 saved. Spearman ρ = 0.147, p = 0.1584


Figure 4 — Heatmap: top 10 siRNA candidates × metrics

In [11]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list
from scipy.spatial.distance import pdist

df_top10 = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')

# ── Prepare numeric matrix ────────────────────────────────────────────────────
# Metrics: dCERES (negative = essential), log2FC, depmap_auc, padj (-log10)
df_top10['neg_log10_padj'] = -np.log10(df_top10['padj'].clip(lower=1e-6))

# For heatmap: invert dCERES so positive = more essential (per know-how guide)
df_top10['essentiality'] = -df_top10['dCERES']

metrics = ['essentiality', 'log2FC', 'depmap_auc', 'neg_log10_padj']
metric_labels = ['Essentiality\n(−dCERES)', 'log₂FC\n(Expression)', 'DepMap\nAUC', '−log₁₀\n(padj)']

# Z-score normalize each metric column
mat = df_top10[metrics].values.astype(float)
mat_z = (mat - mat.mean(axis=0)) / (mat.std(axis=0) + 1e-9)

# Hierarchical clustering on rows
dist = pdist(mat_z, metric='euclidean')
link = linkage(dist, method='ward')
order = leaves_list(link)

genes_ordered   = df_top10['gene'].values[order]
mat_z_ordered   = mat_z[order]
drug_ordered    = df_top10['drug_status'].values[order]

# ── Drug status color strip ───────────────────────────────────────────────────
drug_colors = {'Novel': '#E41A1C', 'Clinical': '#377EB8', 'Preclinical': '#FF7F00'}
drug_strip  = np.array([[mcolors.to_rgb(drug_colors.get(d, '#AAAAAA'))]
                         for d in drug_ordered])

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(11, 8), facecolor='white')
# axes: [left, bottom, width, height]
ax_dend  = fig.add_axes([0.01, 0.12, 0.10, 0.72])   # dendrogram
ax_strip = fig.add_axes([0.12, 0.12, 0.025, 0.72])  # drug strip
ax_heat  = fig.add_axes([0.155, 0.12, 0.62, 0.72])  # heatmap
ax_cbar  = fig.add_axes([0.79, 0.30, 0.02, 0.36])   # colorbar

# Dendrogram
dend = dendrogram(link, orientation='left', ax=ax_dend,
                  color_threshold=0, above_threshold_color='#555555',
                  link_color_func=lambda k: '#555555')
ax_dend.axis('off')

# Drug status strip
ax_strip.imshow(drug_strip, aspect='auto', interpolation='nearest')
ax_strip.set_xticks([])
ax_strip.set_yticks([])
ax_strip.set_title('Drug\nStatus', fontsize=8, pad=4)

# Heatmap — RdBu_r diverging (ColorBrewer)
cmap = plt.cm.RdBu_r
vmax = np.abs(mat_z_ordered).max()
im = ax_heat.imshow(mat_z_ordered, aspect='auto', cmap=cmap,
                    vmin=-vmax, vmax=vmax, interpolation='nearest')

# Gene labels (y-axis)
ax_heat.set_yticks(range(len(genes_ordered)))
ax_heat.set_yticklabels(genes_ordered, fontsize=10.5, fontweight='bold')
ax_heat.yaxis.set_tick_params(length=0)

# Metric labels (x-axis)
ax_heat.set_xticks(range(len(metrics)))
ax_heat.set_xticklabels(metric_labels, fontsize=10, ha='center')
ax_heat.xaxis.set_tick_params(length=0)
ax_heat.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

# Cell value annotations
for i in range(mat_z_ordered.shape[0]):
    for j in range(mat_z_ordered.shape[1]):
        raw_val = mat[order[i], j]
        txt_color = 'white' if abs(mat_z_ordered[i, j]) > 1.2 else '#222222'
        ax_heat.text(j, i, f'{raw_val:.2f}', ha='center', va='center',
                     fontsize=8, color=txt_color)

ax_heat.set_title(
    'Figure 4 — Top 10 siRNA Candidates × Multi-Metric Heatmap\n'
    '[SIMULATED DATA]  Z-score normalized · Hierarchical clustering (Ward)',
    fontsize=11, fontweight='bold', pad=10
)

# Colorbar
cb = plt.colorbar(im, cax=ax_cbar)
cb.set_label('Z-score', fontsize=9)
cb.ax.tick_params(labelsize=8)

# Drug status legend
legend_patches = [Patch(color=v, label=k) for k, v in drug_colors.items()]
fig.legend(handles=legend_patches, title='Drug Status', title_fontsize=9,
           fontsize=9, loc='lower right', bbox_to_anchor=(0.98, 0.02),
           framealpha=0.9)

plt.savefig('/mnt/results/figures/Figure4.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/figures/Figure4.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure 4 saved.")
print("Gene order (clustered):", list(genes_ordered))


Figure 4 saved.
Gene order (clustered): ['POLD1', 'FANCN', 'ATM', 'PLK1', 'CDK1', 'CDC45', 'SPC24', 'ESPL1', 'BUB1B', 'CENPN']


Figure 4 fix — clean -0.00 values and remove unused Preclinical legend entry

In [13]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

df_top10 = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')

df_top10['neg_log10_padj'] = -np.log10(df_top10['padj'].clip(lower=1e-6))
df_top10['essentiality']   = -df_top10['dCERES']

metrics       = ['essentiality', 'log2FC', 'depmap_auc', 'neg_log10_padj']
metric_labels = ['Essentiality\n(−dCERES)', 'log₂FC\n(Expression)', 'DepMap\nAUC', '−log₁₀\n(padj)']

mat   = df_top10[metrics].values.astype(float)
mat_z = (mat - mat.mean(axis=0)) / (mat.std(axis=0) + 1e-9)

dist  = pdist(mat_z, metric='euclidean')
link  = linkage(dist, method='ward')
order = leaves_list(link)

genes_ordered = df_top10['gene'].values[order]
mat_z_ordered = mat_z[order]
drug_ordered  = df_top10['drug_status'].values[order]

# Only include drug categories actually present in top 10
present_drugs = list(dict.fromkeys(drug_ordered))   # preserve order, deduplicate
drug_colors   = {'Novel': '#E41A1C', 'Clinical': '#377EB8', 'Preclinical': '#FF7F00'}
drug_strip    = np.array([[mcolors.to_rgb(drug_colors.get(d, '#AAAAAA'))]
                           for d in drug_ordered])

fig = plt.figure(figsize=(11, 8), facecolor='white')
ax_dend  = fig.add_axes([0.01, 0.12, 0.10, 0.72])
ax_strip = fig.add_axes([0.12, 0.12, 0.025, 0.72])
ax_heat  = fig.add_axes([0.155, 0.12, 0.62, 0.72])
ax_cbar  = fig.add_axes([0.79, 0.30, 0.02, 0.36])

from scipy.cluster.hierarchy import dendrogram
dend = dendrogram(link, orientation='left', ax=ax_dend,
                  color_threshold=0, above_threshold_color='#555555',
                  link_color_func=lambda k: '#555555')
ax_dend.axis('off')

ax_strip.imshow(drug_strip, aspect='auto', interpolation='nearest')
ax_strip.set_xticks([])
ax_strip.set_yticks([])
ax_strip.set_title('Drug\nStatus', fontsize=8, pad=4)

cmap = plt.cm.RdBu_r
vmax = np.abs(mat_z_ordered).max()
im   = ax_heat.imshow(mat_z_ordered, aspect='auto', cmap=cmap,
                      vmin=-vmax, vmax=vmax, interpolation='nearest')

ax_heat.set_yticks(range(len(genes_ordered)))
ax_heat.set_yticklabels(genes_ordered, fontsize=10.5, fontweight='bold')
ax_heat.yaxis.set_tick_params(length=0)
ax_heat.set_xticks(range(len(metrics)))
ax_heat.set_xticklabels(metric_labels, fontsize=10, ha='center')
ax_heat.xaxis.set_tick_params(length=0)
ax_heat.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

# Cell annotations — fix -0.00 by adding 0 before formatting
for i in range(mat_z_ordered.shape[0]):
    for j in range(mat_z_ordered.shape[1]):
        raw_val   = mat[order[i], j] + 0.0   # eliminates -0.0
        txt_color = 'white' if abs(mat_z_ordered[i, j]) > 1.2 else '#222222'
        ax_heat.text(j, i, f'{raw_val:.2f}', ha='center', va='center',
                     fontsize=8, color=txt_color)

ax_heat.set_title(
    'Figure 4 — Top 10 siRNA Candidates × Multi-Metric Heatmap\n'
    '[SIMULATED DATA]  Z-score normalized · Hierarchical clustering (Ward)',
    fontsize=11, fontweight='bold', pad=10
)

cb = plt.colorbar(im, cax=ax_cbar)
cb.set_label('Z-score', fontsize=9)
cb.ax.tick_params(labelsize=8)

# Legend: only categories present in top 10
legend_patches = [Patch(color=drug_colors[d], label=d)
                  for d in present_drugs if d in drug_colors]
fig.legend(handles=legend_patches, title='Drug Status', title_fontsize=9,
           fontsize=9, loc='lower right', bbox_to_anchor=(0.98, 0.02),
           framealpha=0.9)

plt.savefig('/mnt/results/figures/Figure4.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/figures/Figure4.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure 4 (fixed) saved.")
print("Drug categories in top 10:", present_drugs)


Figure 4 (fixed) saved.
Drug categories in top 10: ['Novel', 'Clinical']


Figure 5 — Venn diagram: SL candidates ∩ high expression ∩ no drug

In [15]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import matplotlib.patheffects as pe

df = pd.read_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv')

# Define the three sets
set_SL       = set(df[df['dCERES'] <= -0.10]['gene'])           # SL candidates (dCERES ≤ -0.10)
set_highexpr = set(df[df['log2FC'] >= 1.0]['gene'])             # High expression (log2FC ≥ 1.0)
set_nodrug   = set(df[df['drug_status'] == 'Novel']['gene'])    # No existing drug

# Intersections
only_SL          = set_SL - set_highexpr - set_nodrug
only_highexpr    = set_highexpr - set_SL - set_nodrug
only_nodrug      = set_nodrug - set_SL - set_highexpr
SL_highexpr      = (set_SL & set_highexpr) - set_nodrug
SL_nodrug        = (set_SL & set_nodrug) - set_highexpr
highexpr_nodrug  = (set_highexpr & set_nodrug) - set_SL
triple           = set_SL & set_highexpr & set_nodrug

print(f"Set sizes:")
print(f"  SL candidates (dCERES ≤ -0.10):  {len(set_SL)}")
print(f"  High expression (log2FC ≥ 1.0):  {len(set_highexpr)}")
print(f"  No drug (Novel):                  {len(set_nodrug)}")
print(f"\nIntersections:")
print(f"  SL ∩ HighExpr ∩ NoDrug (triple):  {len(triple)} → {sorted(triple)}")
print(f"  SL ∩ HighExpr only:               {len(SL_highexpr)}")
print(f"  SL ∩ NoDrug only:                 {len(SL_nodrug)}")
print(f"  HighExpr ∩ NoDrug only:           {len(highexpr_nodrug)}")

# ── Draw Venn manually for full control ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8), facecolor='white')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')

# Circle centres and radius
r = 2.8
cx1, cy1 = 3.8, 6.2    # SL candidates (top-left)
cx2, cy2 = 6.2, 6.2    # High expression (top-right)
cx3, cy3 = 5.0, 4.0    # No drug (bottom)

# ColorBrewer Set2 — semi-transparent
c1 = Circle((cx1, cy1), r, color='#66C2A5', alpha=0.38, zorder=2)
c2 = Circle((cx2, cy2), r, color='#FC8D62', alpha=0.38, zorder=2)
c3 = Circle((cx3, cy3), r, color='#8DA0CB', alpha=0.38, zorder=2)
for c in [c1, c2, c3]:
    ax.add_patch(c)

# Circle outlines
for (cx, cy), col in [((cx1,cy1),'#1B9E77'), ((cx2,cy2),'#D95F02'), ((cx3,cy3),'#7570B3')]:
    outline = Circle((cx, cy), r, fill=False, edgecolor=col, lw=2.0, zorder=3)
    ax.add_patch(outline)

# ── Count labels ─────────────────────────────────────────────────────────────
label_kw = dict(ha='center', va='center', fontsize=13, fontweight='bold', zorder=5)

ax.text(2.5, 7.2,  str(len(only_SL)),         color='#1B9E77', **label_kw)
ax.text(7.5, 7.2,  str(len(only_highexpr)),    color='#D95F02', **label_kw)
ax.text(5.0, 2.8,  str(len(only_nodrug)),      color='#7570B3', **label_kw)
ax.text(5.0, 7.0,  str(len(SL_highexpr)),      color='#555555', **label_kw)
ax.text(3.4, 4.8,  str(len(SL_nodrug)),        color='#555555', **label_kw)
ax.text(6.6, 4.8,  str(len(highexpr_nodrug)),  color='#555555', **label_kw)

# Triple intersection — highlighted
ax.text(5.0, 5.8, str(len(triple)),
        ha='center', va='center', fontsize=18, fontweight='bold',
        color='#B2182B', zorder=6,
        path_effects=[pe.withStroke(linewidth=3, foreground='white')])

# Gene names in triple intersection
triple_names = '\n'.join(sorted(triple))
ax.text(5.0, 5.25, triple_names,
        ha='center', va='top', fontsize=8.5, color='#B2182B',
        style='italic', zorder=6,
        path_effects=[pe.withStroke(linewidth=2, foreground='white')])

# ── Set labels ────────────────────────────────────────────────────────────────
ax.text(cx1 - 1.5, cy1 + r + 0.25, f'SL Candidates\n(dCERES ≤ −0.10)\nn = {len(set_SL)}',
        ha='center', va='bottom', fontsize=11, fontweight='bold', color='#1B9E77')
ax.text(cx2 + 1.5, cy2 + r + 0.25, f'High Expression\n(log₂FC ≥ 1.0)\nn = {len(set_highexpr)}',
        ha='center', va='bottom', fontsize=11, fontweight='bold', color='#D95F02')
ax.text(cx3, cy3 - r - 0.35, f'No Existing Drug\n(Novel targets)\nn = {len(set_nodrug)}',
        ha='center', va='top', fontsize=11, fontweight='bold', color='#7570B3')

# Triple intersection callout arrow
ax.annotate(f'Triple intersection\n(highest siRNA value)\nn = {len(triple)}',
            xy=(5.0, 5.7), xytext=(8.2, 8.2),
            fontsize=9.5, color='#B2182B', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#B2182B', lw=1.5),
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF0F0',
                      edgecolor='#B2182B', alpha=0.9), zorder=7)

ax.set_title(
    'Figure 5 — Venn Diagram: SL Candidates ∩ High Expression ∩ No Existing Drug\n'
    '[SIMULATED DATA]  93 DepMap SL Candidates · TCGA LUAD',
    fontsize=12, fontweight='bold', y=0.97
)

plt.tight_layout()
plt.savefig('/mnt/results/figures/Figure5.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/figures/Figure5.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure 5 saved.")


Set sizes:
  SL candidates (dCERES ≤ -0.10):  48
  High expression (log2FC ≥ 1.0):  10
  No drug (Novel):                  58

Intersections:
  SL ∩ HighExpr ∩ NoDrug (triple):  5 → ['BUB1B', 'CDC45', 'CENPN', 'POLD1', 'SPC24']
  SL ∩ HighExpr only:               3
  SL ∩ NoDrug only:                 29
  HighExpr ∩ NoDrug only:           1
Figure 5 saved.


Figure 5 fix — improved spacing, title placement, label positions

In [17]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import matplotlib.patheffects as pe

df = pd.read_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv')

set_SL       = set(df[df['dCERES'] <= -0.10]['gene'])
set_highexpr = set(df[df['log2FC'] >= 1.0]['gene'])
set_nodrug   = set(df[df['drug_status'] == 'Novel']['gene'])

only_SL         = set_SL - set_highexpr - set_nodrug
only_highexpr   = set_highexpr - set_SL - set_nodrug
only_nodrug     = set_nodrug - set_SL - set_highexpr
SL_highexpr     = (set_SL & set_highexpr) - set_nodrug
SL_nodrug       = (set_SL & set_nodrug) - set_highexpr
highexpr_nodrug = (set_highexpr & set_nodrug) - set_SL
triple          = set_SL & set_highexpr & set_nodrug

fig, ax = plt.subplots(figsize=(11, 9), facecolor='white')
ax.set_xlim(0, 11)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')

# Title at very top — compact single block
fig.text(0.5, 0.96,
         'Figure 5 — Venn Diagram: SL Candidates ∩ High Expression ∩ No Existing Drug',
         ha='center', va='top', fontsize=12, fontweight='bold')
fig.text(0.5, 0.925,
         '[SIMULATED DATA]  93 DepMap SL Candidates · TCGA LUAD',
         ha='center', va='top', fontsize=10, color='#555555', style='italic')

# Circles — shifted down to give title room
r = 2.7
cx1, cy1 = 3.9, 5.8    # SL candidates (left)
cx2, cy2 = 7.1, 5.8    # High expression (right)
cx3, cy3 = 5.5, 3.6    # No drug (bottom)

c1 = Circle((cx1, cy1), r, color='#66C2A5', alpha=0.35, zorder=2)
c2 = Circle((cx2, cy2), r, color='#FC8D62', alpha=0.35, zorder=2)
c3 = Circle((cx3, cy3), r, color='#8DA0CB', alpha=0.35, zorder=2)
for c in [c1, c2, c3]:
    ax.add_patch(c)
for (cx, cy), col in [((cx1,cy1),'#1B9E77'), ((cx2,cy2),'#D95F02'), ((cx3,cy3),'#7570B3')]:
    ax.add_patch(Circle((cx, cy), r, fill=False, edgecolor=col, lw=2.0, zorder=3))

# Count labels
lkw = dict(ha='center', va='center', fontsize=12, fontweight='bold', zorder=5)
ax.text(2.2, 6.5,  str(len(only_SL)),        color='#1B9E77', **lkw)
ax.text(8.8, 6.5,  str(len(only_highexpr)),  color='#D95F02', **lkw)
ax.text(5.5, 1.8,  str(len(only_nodrug)),    color='#7570B3', **lkw)
ax.text(5.5, 6.8,  str(len(SL_highexpr)),    color='#444444', **lkw)
ax.text(3.5, 4.3,  str(len(SL_nodrug)),      color='#444444', **lkw)
ax.text(7.5, 4.3,  str(len(highexpr_nodrug)),color='#444444', **lkw)

# Triple intersection — count above, genes below
ax.text(5.5, 5.5, str(len(triple)),
        ha='center', va='center', fontsize=20, fontweight='bold',
        color='#B2182B', zorder=6,
        path_effects=[pe.withStroke(linewidth=3, foreground='white')])
triple_names = '\n'.join(sorted(triple))
ax.text(5.5, 4.95, triple_names,
        ha='center', va='top', fontsize=8, color='#B2182B',
        style='italic', zorder=6,
        path_effects=[pe.withStroke(linewidth=2, foreground='white')])

# Set labels — well below title, outside circles
ax.text(cx1 - 2.2, cy1 + 0.3,
        f'SL Candidates\n(dCERES ≤ −0.10)\nn = {len(set_SL)}',
        ha='center', va='center', fontsize=10.5, fontweight='bold', color='#1B9E77')
ax.text(cx2 + 2.2, cy2 + 0.3,
        f'High Expression\n(log₂FC ≥ 1.0)\nn = {len(set_highexpr)}',
        ha='center', va='center', fontsize=10.5, fontweight='bold', color='#D95F02')
ax.text(cx3, cy3 - r - 0.55,
        f'No Existing Drug\n(Novel targets)\nn = {len(set_nodrug)}',
        ha='center', va='top', fontsize=10.5, fontweight='bold', color='#7570B3')

# Callout — arrow from outside pointing to triple zone
ax.annotate(f'Triple intersection\n(highest siRNA value)\nn = {len(triple)}',
            xy=(5.5, 5.45), xytext=(9.5, 8.5),
            fontsize=9, color='#B2182B', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#B2182B', lw=1.5,
                            connectionstyle='arc3,rad=-0.2'),
            bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFF0F0',
                      edgecolor='#B2182B', alpha=0.92), zorder=7)

plt.savefig('/mnt/results/figures/Figure5.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/figures/Figure5.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure 5 (fixed) saved.")


Figure 5 (fixed) saved.


C1: Compute actual median VAF and TMB from simulated dataset

In [19]:

import pandas as pd
import numpy as np

df = pd.read_csv('/mnt/results/data/SIMULATED_TCGA_LUAD_TP53_mutations.csv')

# Compute median VAF (mutant only — WT has NaN)
df_mut = df[df['tp53_status'] == 'Mutant']
df_wt  = df[df['tp53_status'] == 'WT']

median_vaf_mut = df_mut['vaf'].median()
median_tmb_mut = df_mut['tmb'].median()
median_tmb_wt  = df_wt['tmb'].median()
mean_vaf_mut   = df_mut['vaf'].mean()
std_vaf_mut    = df_mut['vaf'].std()

print("=== C1: Computed Statistics ===")
print(f"Median VAF (TP53-mutant):       {median_vaf_mut:.3f}")
print(f"Mean VAF (TP53-mutant):         {mean_vaf_mut:.3f} ± {std_vaf_mut:.3f}")
print(f"Median TMB (TP53-mutant):       {median_tmb_mut:.2f} mut/Mb")
print(f"Median TMB (TP53-WT):           {median_tmb_wt:.2f} mut/Mb")
print(f"TMB ratio (mut/WT):             {median_tmb_mut/median_tmb_wt:.2f}x")

# Mutation type counts for verification
print("\nMutation type counts:")
print(df_mut['mutation_type'].value_counts())
print(f"\nTotal mutant samples: {len(df_mut)}")
print(f"Total WT samples:     {len(df_wt)}")


=== C1: Computed Statistics ===
Median VAF (TP53-mutant):       0.627
Mean VAF (TP53-mutant):         0.612 ± 0.207
Median TMB (TP53-mutant):       16.09 mut/Mb
Median TMB (TP53-WT):           9.46 mut/Mb
TMB ratio (mut/WT):             1.70x

Mutation type counts:
mutation_type
Missense             180
Nonsense              52
Frameshift            34
Splice Site           24
In-frame Deletion      5
Name: count, dtype: int64

Total mutant samples: 295
Total WT samples:     271


C2: Replace Bonferroni with BH/FDR correction, update CSVs

In [21]:

import pandas as pd
import numpy as np
from statsmodels.stats.multitest import multipletests

df = pd.read_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv')

print("=== Before BH/FDR correction ===")
print(f"Genes with padj <= 0.05 (old Bonferroni): {(df['padj'] <= 0.05).sum()}")
print(f"padj range: {df['padj'].min():.4f} – {df['padj'].max():.4f}")

# Apply BH/FDR correction to the raw p-values
reject, padj_bh, _, _ = multipletests(df['pvalue'], alpha=0.05, method='fdr_bh')

df['padj'] = np.round(padj_bh, 4)

print("\n=== After BH/FDR correction ===")
print(f"Genes with padj <= 0.05 (BH/FDR):         {(df['padj'] <= 0.05).sum()}")
print(f"padj range: {df['padj'].min():.4f} – {df['padj'].max():.4f}")

# Show top candidates
top5_genes = ['SPC24', 'BUB1B', 'CDC45', 'PLK1', 'CDK1']
print("\nTop 5 validated candidates after BH/FDR:")
print(df[df['gene'].isin(top5_genes)][['gene','pvalue','padj','dCERES','log2FC']].to_string())

# Significant SL hits under new correction
sig = df[(df['padj'] <= 0.05) & (df['dCERES'] <= -0.10)]
print(f"\nSignificant SL hits (padj<=0.05 AND dCERES<=-0.10): {len(sig)}")
print(sig[['gene','dCERES','log2FC','padj','drug_status']].to_string())

# Save updated CSV
df.to_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv', index=False)
print("\nUpdated SIMULATED_DepMap_93_SL_candidates.csv saved.")


=== Before BH/FDR correction ===
Genes with padj <= 0.05 (old Bonferroni): 5
padj range: 0.0009 – 1.0000

=== After BH/FDR correction ===
Genes with padj <= 0.05 (BH/FDR):         6
padj range: 0.0093 – 0.1492

Top 5 validated candidates after BH/FDR:
    gene  pvalue    padj  dCERES  log2FC
0  SPC24  0.0008  0.0248  -0.175    1.13
1  BUB1B  0.0021  0.0391  -0.119    1.12
2  CDC45  0.0012  0.0279  -0.144    1.26
3   PLK1  0.0001  0.0093  -0.239    1.03
4   CDK1  0.0003  0.0139  -0.201    1.00

Significant SL hits (padj<=0.05 AND dCERES<=-0.10): 5
    gene  dCERES  log2FC    padj drug_status
0  SPC24  -0.175    1.13  0.0248       Novel
1  BUB1B  -0.119    1.12  0.0391       Novel
2  CDC45  -0.144    1.26  0.0279       Novel
3   PLK1  -0.239    1.03  0.0093    Clinical
4   CDK1  -0.201    1.00  0.0139    Clinical

Updated SIMULATED_DepMap_93_SL_candidates.csv saved.


C6: Z-score normalize SL composite score, recompute top-10, update CSV

In [23]:

import pandas as pd
import numpy as np
from scipy.stats import zscore

df = pd.read_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv')

# Old unnormalized score for comparison
df['sl_score_old'] = -df['dCERES'] + df['log2FC']

# C6 fix: Z-score normalize both components before combining
# Per user instruction: SL_score = zscore(-dCERES) + zscore(log2FC)
df['z_neg_dceres'] = zscore(-df['dCERES'])   # positive = more essential
df['z_log2fc']     = zscore(df['log2FC'])    # positive = more overexpressed
df['sl_score']     = df['z_neg_dceres'] + df['z_log2fc']

print("=== Scale comparison (justification for Z-score normalization) ===")
print(f"dCERES range:       {df['dCERES'].min():.3f} to {df['dCERES'].max():.3f}  (span={df['dCERES'].max()-df['dCERES'].min():.3f})")
print(f"log2FC range:       {df['log2FC'].min():.3f} to {df['log2FC'].max():.3f}  (span={df['log2FC'].max()-df['log2FC'].min():.3f})")
print(f"z_neg_dceres range: {df['z_neg_dceres'].min():.3f} to {df['z_neg_dceres'].max():.3f}")
print(f"z_log2fc range:     {df['z_log2fc'].min():.3f} to {df['z_log2fc'].max():.3f}")

# New top-10
top10_new = df.nlargest(10, 'sl_score').reset_index(drop=True)
top10_new.to_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv', index=False)

print("\n=== New top-10 (Z-score normalized SL score) ===")
print(top10_new[['gene','dCERES','log2FC','sl_score','drug_status','depmap_auc','padj']].to_string())

# Check if top-5 validated candidates are still in top-10
top5 = ['SPC24', 'BUB1B', 'CDC45', 'PLK1', 'CDK1']
in_top10 = [g for g in top5 if g in top10_new['gene'].values]
print(f"\nValidated top-5 still in new top-10: {in_top10}")
missing = [g for g in top5 if g not in top10_new['gene'].values]
if missing:
    print(f"WARNING — dropped from top-10: {missing}")
    print(df[df['gene'].isin(missing)][['gene','sl_score','dCERES','log2FC']].to_string())


=== Scale comparison (justification for Z-score normalization) ===
dCERES range:       -0.239 to 0.036  (span=0.275)
log2FC range:       -0.402 to 1.260  (span=1.662)
z_neg_dceres range: -1.642 to 1.758
z_log2fc range:     -2.364 to 2.090

=== New top-10 (Z-score normalized SL score) ===
    gene  dCERES  log2FC  sl_score drug_status  depmap_auc    padj
0   PLK1  -0.239   1.030  3.231747    Clinical       0.820  0.0093
1  POLD1  -0.208   1.157  3.188682       Novel       0.490  0.1448
2  FANCN  -0.213   0.977  2.768204       Novel       0.523  0.1395
3  ESPL1  -0.153   1.235  2.717525    Clinical       0.664  0.1448
4  SPC24  -0.175   1.130  2.708240       Novel       0.740  0.0248
5   CDK1  -0.201   1.000  2.681434    Clinical       0.800  0.0139
6  CDC45  -0.144   1.260  2.673214       Novel       0.760  0.0279
7  NDC80  -0.204   0.916  2.493455       Novel       0.575  0.1448
8    ATM  -0.191   0.936  2.386280       Novel       0.497  0.1448
9  CENPN  -0.141   1.098  2.202034       

Regenerate Figure 2 with BH/FDR-corrected padj values

In [25]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from adjustText import adjust_text

df = pd.read_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv')

df['neg_log10_padj'] = -np.log10(df['padj'].clip(lower=1e-6))

padj_thresh   = 0.05
dceres_thresh = -0.10

def classify(row):
    if row['padj'] <= padj_thresh and row['dCERES'] <= dceres_thresh:
        return 'Significant SL'
    elif row['padj'] <= padj_thresh:
        return 'Significant only'
    elif row['dCERES'] <= dceres_thresh:
        return 'dCERES only'
    else:
        return 'Not significant'

df['category'] = df.apply(classify, axis=1)
print("Category counts (BH/FDR):")
print(df['category'].value_counts())

cat_colors = {
    'Significant SL':   '#E41A1C',
    'Significant only': '#FF7F00',
    'dCERES only':      '#377EB8',
    'Not significant':  '#AAAAAA',
}
cat_sizes = {'Significant SL': 55, 'Significant only': 40,
             'dCERES only': 40, 'Not significant': 25}

top5 = ['SPC24', 'BUB1B', 'CDC45', 'PLK1', 'CDK1']

fig, ax = plt.subplots(figsize=(10, 7), facecolor='white')

for cat, grp in df.groupby('category'):
    ax.scatter(grp['dCERES'], grp['neg_log10_padj'],
               c=cat_colors[cat], s=cat_sizes[cat],
               alpha=0.80, linewidths=0.4, edgecolors='white',
               label=cat, zorder=3)

ax.axvline(dceres_thresh, color='#555555', lw=1.0, ls='--', alpha=0.7)
ax.axhline(-np.log10(padj_thresh), color='#555555', lw=1.0, ls='--', alpha=0.7)

texts = []
for _, row in df[df['gene'].isin(top5)].iterrows():
    drug_label = '★' if row['drug_status'] == 'Clinical' else '◆'
    texts.append(
        ax.text(row['dCERES'], row['neg_log10_padj'],
                f" {drug_label} {row['gene']}",
                fontsize=9.5, fontweight='bold', color='#1A1A1A')
    )
adjust_text(texts, ax=ax,
            arrowprops=dict(arrowstyle='->', color='#444444', lw=0.8))

ax.text(dceres_thresh + 0.003, ax.get_ylim()[1]*0.97,
        'dCERES = −0.10', fontsize=8, color='#555555', va='top')
ax.text(ax.get_xlim()[0] + 0.002, -np.log10(padj_thresh) + 0.05,
        'padj = 0.05 (BH/FDR)', fontsize=8, color='#555555')

ax.set_xlabel('Differential CERES Score (TP53-mut − WT)\n← More essential in TP53-mut | Less essential →',
              fontsize=11)
ax.set_ylabel('−log₁₀(adjusted p-value, BH/FDR)', fontsize=11)
ax.set_title('Figure 2 — Differential Essentiality Volcano Plot (DepMap)\n'
             '[SIMULATED DATA]  93 SL Candidates · BH/FDR correction · TP53-Mutant vs WT LUAD',
             fontsize=12, fontweight='bold')

legend = ax.legend(title='Category', fontsize=9, title_fontsize=9,
                   framealpha=0.9, loc='upper right')

from matplotlib.lines import Line2D
extra = [Line2D([0],[0], marker='$★$', color='w', markerfacecolor='#333',
                markersize=10, label='Clinical drug exists'),
         Line2D([0],[0], marker='$◆$', color='w', markerfacecolor='#333',
                markersize=10, label='Novel (no drug)')]
ax.add_artist(legend)
ax.legend(handles=extra, loc='lower left', fontsize=9, framealpha=0.9,
          title='Drug Status', title_fontsize=9)

ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('white')
plt.tight_layout()

plt.savefig('/mnt/results/figures/Figure2.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/figures/Figure2.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure 2 regenerated with BH/FDR correction.")


Category counts (BH/FDR):
category
Not significant     44
dCERES only         43
Significant SL       5
Significant only     1
Name: count, dtype: int64
Figure 2 regenerated with BH/FDR correction.


Regenerate Figure 4 heatmap with Z-score normalized top-10

In [27]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import pdist

df_top10 = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')

print("New top-10 genes:", list(df_top10['gene']))
print("Drug categories present:", df_top10['drug_status'].unique())

df_top10['neg_log10_padj'] = -np.log10(df_top10['padj'].clip(lower=1e-6))
df_top10['essentiality']   = -df_top10['dCERES']   # inverted per know-how guide

metrics       = ['essentiality', 'log2FC', 'depmap_auc', 'neg_log10_padj']
metric_labels = ['Essentiality\n(−dCERES)', 'log₂FC\n(Expression)', 'DepMap\nAUC', '−log₁₀\n(padj)']

mat   = df_top10[metrics].values.astype(float)
mat_z = (mat - mat.mean(axis=0)) / (mat.std(axis=0) + 1e-9)

dist  = pdist(mat_z, metric='euclidean')
link  = linkage(dist, method='ward')
order = leaves_list(link)

genes_ordered = df_top10['gene'].values[order]
mat_z_ordered = mat_z[order]
drug_ordered  = df_top10['drug_status'].values[order]

# Mark validated SL hits (padj <= 0.05 AND dCERES <= -0.10)
validated_sl = set(df_top10[
    (df_top10['padj'] <= 0.05) & (df_top10['dCERES'] <= -0.10)
]['gene'])
print(f"Validated SL hits in top-10: {validated_sl}")

present_drugs = list(dict.fromkeys(drug_ordered))
drug_colors   = {'Novel': '#E41A1C', 'Clinical': '#377EB8', 'Preclinical': '#FF7F00'}
drug_strip    = np.array([[mcolors.to_rgb(drug_colors.get(d, '#AAAAAA'))]
                           for d in drug_ordered])

fig = plt.figure(figsize=(11, 8), facecolor='white')
ax_dend  = fig.add_axes([0.01, 0.12, 0.10, 0.72])
ax_strip = fig.add_axes([0.12, 0.12, 0.025, 0.72])
ax_heat  = fig.add_axes([0.155, 0.12, 0.62, 0.72])
ax_cbar  = fig.add_axes([0.79, 0.30, 0.02, 0.36])

dend = dendrogram(link, orientation='left', ax=ax_dend,
                  color_threshold=0, above_threshold_color='#555555',
                  link_color_func=lambda k: '#555555')
ax_dend.axis('off')

ax_strip.imshow(drug_strip, aspect='auto', interpolation='nearest')
ax_strip.set_xticks([])
ax_strip.set_yticks([])
ax_strip.set_title('Drug\nStatus', fontsize=8, pad=4)

cmap = plt.cm.RdBu_r
vmax = np.abs(mat_z_ordered).max()
im   = ax_heat.imshow(mat_z_ordered, aspect='auto', cmap=cmap,
                      vmin=-vmax, vmax=vmax, interpolation='nearest')

# Gene labels — mark validated SL hits with asterisk
yticklabels = []
for g in genes_ordered:
    label = f"★ {g}" if g in validated_sl else g
    yticklabels.append(label)

ax_heat.set_yticks(range(len(genes_ordered)))
ax_heat.set_yticklabels(yticklabels, fontsize=10.5, fontweight='bold')
ax_heat.yaxis.set_tick_params(length=0)
ax_heat.set_xticks(range(len(metrics)))
ax_heat.set_xticklabels(metric_labels, fontsize=10, ha='center')
ax_heat.xaxis.set_tick_params(length=0)
ax_heat.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

# Cell value annotations
for i in range(mat_z_ordered.shape[0]):
    for j in range(mat_z_ordered.shape[1]):
        raw_val   = mat[order[i], j] + 0.0
        txt_color = 'white' if abs(mat_z_ordered[i, j]) > 1.2 else '#222222'
        ax_heat.text(j, i, f'{raw_val:.2f}', ha='center', va='center',
                     fontsize=8, color=txt_color)

ax_heat.set_title(
    'Figure 4 — Top 10 siRNA Candidates × Multi-Metric Heatmap\n'
    '[SIMULATED DATA]  Z-score normalized · Ward clustering · ★ = Validated SL (padj≤0.05, dCERES≤−0.10)',
    fontsize=10.5, fontweight='bold', pad=10
)

cb = plt.colorbar(im, cax=ax_cbar)
cb.set_label('Z-score', fontsize=9)
cb.ax.tick_params(labelsize=8)

legend_patches = [Patch(color=drug_colors[d], label=d)
                  for d in present_drugs if d in drug_colors]
fig.legend(handles=legend_patches, title='Drug Status', title_fontsize=9,
           fontsize=9, loc='lower right', bbox_to_anchor=(0.98, 0.02),
           framealpha=0.9)

plt.savefig('/mnt/results/figures/Figure4.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/figures/Figure4.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure 4 regenerated with Z-score normalized top-10.")


New top-10 genes: ['PLK1', 'POLD1', 'FANCN', 'ESPL1', 'SPC24', 'CDK1', 'CDC45', 'NDC80', 'ATM', 'CENPN']
Drug categories present: ['Clinical' 'Novel']
Validated SL hits in top-10: {'CDK1', 'PLK1', 'CDC45', 'SPC24'}
Figure 4 regenerated with Z-score normalized top-10.


Figure 4 fix — replace Unicode star with ASCII [*] to avoid font substitution

In [29]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram
from scipy.spatial.distance import pdist

df_top10 = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')

df_top10['neg_log10_padj'] = -np.log10(df_top10['padj'].clip(lower=1e-6))
df_top10['essentiality']   = -df_top10['dCERES']

metrics       = ['essentiality', 'log2FC', 'depmap_auc', 'neg_log10_padj']
metric_labels = ['Essentiality\n(-dCERES)', 'log2FC\n(Expression)', 'DepMap\nAUC', '-log10\n(padj)']

mat   = df_top10[metrics].values.astype(float)
mat_z = (mat - mat.mean(axis=0)) / (mat.std(axis=0) + 1e-9)

dist  = pdist(mat_z, metric='euclidean')
link  = linkage(dist, method='ward')
order = leaves_list(link)

genes_ordered = df_top10['gene'].values[order]
mat_z_ordered = mat_z[order]
drug_ordered  = df_top10['drug_status'].values[order]

validated_sl = set(df_top10[
    (df_top10['padj'] <= 0.05) & (df_top10['dCERES'] <= -0.10)
]['gene'])

present_drugs = list(dict.fromkeys(drug_ordered))
drug_colors   = {'Novel': '#E41A1C', 'Clinical': '#377EB8'}
drug_strip    = np.array([[mcolors.to_rgb(drug_colors.get(d, '#AAAAAA'))]
                           for d in drug_ordered])

fig = plt.figure(figsize=(11, 8), facecolor='white')
ax_dend  = fig.add_axes([0.01, 0.12, 0.10, 0.72])
ax_strip = fig.add_axes([0.12, 0.12, 0.025, 0.72])
ax_heat  = fig.add_axes([0.155, 0.12, 0.62, 0.72])
ax_cbar  = fig.add_axes([0.79, 0.30, 0.02, 0.36])

dendrogram(link, orientation='left', ax=ax_dend,
           color_threshold=0, above_threshold_color='#555555',
           link_color_func=lambda k: '#555555')
ax_dend.axis('off')

ax_strip.imshow(drug_strip, aspect='auto', interpolation='nearest')
ax_strip.set_xticks([])
ax_strip.set_yticks([])
ax_strip.set_title('Drug\nStatus', fontsize=8, pad=4)

cmap = plt.cm.RdBu_r
vmax = np.abs(mat_z_ordered).max()
im   = ax_heat.imshow(mat_z_ordered, aspect='auto', cmap=cmap,
                      vmin=-vmax, vmax=vmax, interpolation='nearest')

# Use [*] instead of Unicode star to avoid font substitution
yticklabels = [f"[*] {g}" if g in validated_sl else f"    {g}"
               for g in genes_ordered]

ax_heat.set_yticks(range(len(genes_ordered)))
ax_heat.set_yticklabels(yticklabels, fontsize=10.5, fontweight='bold',
                        fontfamily='monospace')
ax_heat.yaxis.set_tick_params(length=0)
ax_heat.set_xticks(range(len(metrics)))
ax_heat.set_xticklabels(metric_labels, fontsize=10, ha='center')
ax_heat.xaxis.set_tick_params(length=0)
ax_heat.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

for i in range(mat_z_ordered.shape[0]):
    for j in range(mat_z_ordered.shape[1]):
        raw_val   = mat[order[i], j] + 0.0
        txt_color = 'white' if abs(mat_z_ordered[i, j]) > 1.2 else '#222222'
        ax_heat.text(j, i, f'{raw_val:.2f}', ha='center', va='center',
                     fontsize=8, color=txt_color)

ax_heat.set_title(
    'Figure 4 - Top 10 siRNA Candidates x Multi-Metric Heatmap\n'
    '[SIMULATED DATA]  Z-score normalized  |  Ward clustering  |  [*] = Validated SL (padj<=0.05, dCERES<=-0.10)',
    fontsize=10.5, fontweight='bold', pad=10
)

cb = plt.colorbar(im, cax=ax_cbar)
cb.set_label('Z-score', fontsize=9)
cb.ax.tick_params(labelsize=8)

legend_patches = [Patch(color=drug_colors[d], label=d)
                  for d in present_drugs if d in drug_colors]
fig.legend(handles=legend_patches, title='Drug Status', title_fontsize=9,
           fontsize=9, loc='lower right', bbox_to_anchor=(0.98, 0.02),
           framealpha=0.9)

plt.savefig('/mnt/results/figures/Figure4.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig('/mnt/results/figures/Figure4.svg', bbox_inches='tight', facecolor='white')
plt.close()
print("Figure 4 fixed (ASCII labels).")
print("Validated SL in top-10:", validated_sl)


Figure 4 fixed (ASCII labels).
Validated SL in top-10: {'CDK1', 'PLK1', 'CDC45', 'SPC24'}


C5: Add venn_category column to DepMap CSV, identify 20 unclassified genes

In [31]:

import pandas as pd
import numpy as np

df = pd.read_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv')

# Reproduce the three Venn set membership criteria
is_sl       = df['dCERES'] <= -0.10
is_highexpr = df['log2FC'] >= 1.0
is_nodrug   = df['drug_status'] == 'Novel'

# Assign Venn category
def venn_cat(row):
    sl  = row['dCERES'] <= -0.10
    he  = row['log2FC'] >= 1.0
    nd  = row['drug_status'] == 'Novel'
    if sl and he and nd:   return 'Triple (SL + HighExpr + NoDrug)'
    elif sl and he:        return 'SL + HighExpr'
    elif sl and nd:        return 'SL + NoDrug'
    elif he and nd:        return 'HighExpr + NoDrug'
    elif sl:               return 'SL only'
    elif he:               return 'HighExpr only'
    elif nd:               return 'NoDrug only'
    else:                  return 'Unclassified (none of three criteria)'

df['venn_category'] = df.apply(venn_cat, axis=1)

print("=== Venn category distribution ===")
print(df['venn_category'].value_counts())
print(f"\nTotal genes: {len(df)}")
print(f"Genes in Venn diagram: {(df['venn_category'] != 'Unclassified (none of three criteria)').sum()}")
print(f"Unclassified genes:    {(df['venn_category'] == 'Unclassified (none of three criteria)').sum()}")

unclassified = df[df['venn_category'] == 'Unclassified (none of three criteria)']
print(f"\nUnclassified gene list ({len(unclassified)}):")
print(list(unclassified['gene']))

# Verify Venn counts match original
triple = df[df['venn_category'] == 'Triple (SL + HighExpr + NoDrug)']
print(f"\nTriple intersection genes: {list(triple['gene'])}")

# Save updated CSV
df.to_csv('/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv', index=False)
print("\nUpdated CSV saved with venn_category column.")


=== Venn category distribution ===
venn_category
SL + NoDrug                              29
NoDrug only                              23
Unclassified (none of three criteria)    20
SL only                                  11
Triple (SL + HighExpr + NoDrug)           5
SL + HighExpr                             3
HighExpr only                             1
HighExpr + NoDrug                         1
Name: count, dtype: int64

Total genes: 93
Genes in Venn diagram: 73
Unclassified genes:    20

Unclassified gene list (20):
['AURKB', 'CENPE', 'CHEK1', 'CDC20', 'CDCA8', 'SGOL1', 'PTTG1', 'CCNB1', 'CDC25C', 'FANCD2', 'PCNA', 'RFC4', 'RFC5', 'MCM7', 'CLASPIN', 'RECQL4', 'HELQ', 'FANCM', 'FANCC', 'FANCO']

Triple intersection genes: ['SPC24', 'BUB1B', 'CDC45', 'CENPN', 'POLD1']

Updated CSV saved with venn_category column.


Confirm actual top-10 genes by Z-score composite SL score

In [33]:

import pandas as pd

df = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')
print("Top-10 columns:", list(df.columns))
print("\nTop-10 genes (ranked by sl_score):")
print(df[['gene', 'dCERES', 'log2FC', 'drug_status', 'sl_score']].to_string(index=False))


Top-10 columns: ['gene', 'dCERES', 'log2FC', 'drug_status', 'depmap_auc', 'pvalue', 'padj', 'sl_score_old', 'z_neg_dceres', 'z_log2fc', 'sl_score']

Top-10 genes (ranked by sl_score):
 gene  dCERES  log2FC drug_status  sl_score
 PLK1  -0.239   1.030    Clinical  3.231747
POLD1  -0.208   1.157       Novel  3.188682
FANCN  -0.213   0.977       Novel  2.768204
ESPL1  -0.153   1.235    Clinical  2.717525
SPC24  -0.175   1.130       Novel  2.708240
 CDK1  -0.201   1.000    Clinical  2.681434
CDC45  -0.144   1.260       Novel  2.673214
NDC80  -0.204   0.916       Novel  2.493455
  ATM  -0.191   0.936       Novel  2.386280
CENPN  -0.141   1.098       Novel  2.202034


Run Ward hierarchical clustering on top-10 Z-score normalized metrics to get true cluster assignments

In [35]:

import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import zscore

df = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')

# Confirm top-10 order
df_top10 = df.sort_values('sl_score', ascending=False).head(10).copy()
print("Top-10 genes in rank order:")
print(df_top10[['gene', 'dCERES', 'log2FC', 'drug_status', 'sl_score']].to_string(index=False))

# Build the 4-metric matrix used for the heatmap (same as Figure 4 code)
metrics = {
    'Essentiality\n(-dCERES)': -df_top10['dCERES'],
    'Expression\n(log2FC)':     df_top10['log2FC'],
    'CERES\nSelectivity':       df_top10['depmap_auc'],
    '-log10(padj)':             -np.log10(df_top10['padj'])
}
mat = pd.DataFrame(metrics, index=df_top10['gene'])

# Z-score normalize each column
mat_z = mat.apply(zscore)

# Ward clustering with k=2 (same as Figure 4)
Z = linkage(mat_z, method='ward', metric='euclidean')
labels = fcluster(Z, t=2, criterion='maxclust')

df_top10 = df_top10.copy()
df_top10['cluster'] = labels

print("\nCluster assignments (Ward, k=2):")
for c in sorted(df_top10['cluster'].unique()):
    genes = df_top10[df_top10['cluster'] == c][['gene', 'drug_status', 'dCERES', 'log2FC']].values
    print(f"\nCluster {c}:")
    for g in genes:
        print(f"  {g[0]:8s}  drug={g[1]:10s}  dCERES={g[2]:.3f}  log2FC={g[3]:.3f}")


Top-10 genes in rank order:
 gene  dCERES  log2FC drug_status  sl_score
 PLK1  -0.239   1.030    Clinical  3.231747
POLD1  -0.208   1.157       Novel  3.188682
FANCN  -0.213   0.977       Novel  2.768204
ESPL1  -0.153   1.235    Clinical  2.717525
SPC24  -0.175   1.130       Novel  2.708240
 CDK1  -0.201   1.000    Clinical  2.681434
CDC45  -0.144   1.260       Novel  2.673214
NDC80  -0.204   0.916       Novel  2.493455
  ATM  -0.191   0.936       Novel  2.386280
CENPN  -0.141   1.098       Novel  2.202034
Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 28, in <module>
  File "/opt/conda/lib/python3.11/site-packages/scipy/cluster/hierarchy.py", line 1030, in linkage
    raise ValueError("The condensed distance matrix must contain only "
ValueError: The condensed distance matrix must contain only finite values.


Run Ward clustering with NaN-safe zscore normalization on top-10 metrics

In [37]:

import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.stats import zscore

df = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')
df_top10 = df.sort_values('sl_score', ascending=False).head(10).copy().reset_index(drop=True)

# Check for constant columns
print("padj unique values:", df_top10['padj'].unique())
print("depmap_auc unique values:", df_top10['depmap_auc'].unique())

# Build 4-metric matrix
mat = pd.DataFrame({
    'Essentiality': -df_top10['dCERES'],
    'Expression':    df_top10['log2FC'],
    'Selectivity':   df_top10['depmap_auc'],
    'Significance':  -np.log10(df_top10['padj'])
}, index=df_top10['gene'])

print("\nRaw matrix:\n", mat.round(3))

# Z-score normalize — replace NaN (constant cols) with 0
mat_z = mat.apply(lambda col: zscore(col) if col.std() > 0 else pd.Series(np.zeros(len(col)), index=col.index))
print("\nZ-score matrix:\n", mat_z.round(3))
print("\nAny NaN?", mat_z.isna().any().any())

# Ward clustering k=2
Z = linkage(mat_z.values, method='ward', metric='euclidean')
labels = fcluster(Z, t=2, criterion='maxclust')
df_top10['cluster'] = labels

print("\n=== CLUSTER ASSIGNMENTS (Ward, k=2) ===")
for c in sorted(df_top10['cluster'].unique()):
    sub = df_top10[df_top10['cluster'] == c]
    genes = sub['gene'].tolist()
    statuses = sub['drug_status'].tolist()
    dceres = sub['dCERES'].tolist()
    log2fc = sub['log2FC'].tolist()
    print(f"\nCluster {c} (n={len(sub)}):")
    for g, s, d, l in zip(genes, statuses, dceres, log2fc):
        print(f"  {g:8s}  {s:10s}  dCERES={d:.3f}  log2FC={l:.3f}")


padj unique values: [0.0093 0.1448 0.1395 0.0248 0.0139 0.0279 0.0593]
depmap_auc unique values: [0.82  0.49  0.523 0.664 0.74  0.8   0.76  0.575 0.497 0.766]

Raw matrix:
        Essentiality  Expression  Selectivity  Significance
gene                                                      
PLK1            NaN         NaN          NaN           NaN
POLD1           NaN         NaN          NaN           NaN
FANCN           NaN         NaN          NaN           NaN
ESPL1           NaN         NaN          NaN           NaN
SPC24           NaN         NaN          NaN           NaN
CDK1            NaN         NaN          NaN           NaN
CDC45           NaN         NaN          NaN           NaN
NDC80           NaN         NaN          NaN           NaN
ATM             NaN         NaN          NaN           NaN
CENPN           NaN         NaN          NaN           NaN

Z-score matrix:
        Essentiality  Expression  Selectivity  Significance
gene                                      

CRITICAL-1: Diagnose NaN bug — print columns, dtypes, head before normalization

In [1]:

import pandas as pd
import numpy as np

df = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')

print("=== COLUMNS ===")
print(df.columns.tolist())

print("\n=== DTYPES ===")
print(df.dtypes)

print("\n=== HEAD (5 rows) ===")
print(df.head())

print("\n=== NULL COUNTS ===")
print(df.isnull().sum())


=== COLUMNS ===
['gene', 'dCERES', 'log2FC', 'drug_status', 'depmap_auc', 'pvalue', 'padj', 'sl_score_old', 'z_neg_dceres', 'z_log2fc', 'sl_score']

=== DTYPES ===
gene             object
dCERES          float64
log2FC          float64
drug_status      object
depmap_auc      float64
pvalue          float64
padj            float64
sl_score_old    float64
z_neg_dceres    float64
z_log2fc        float64
sl_score        float64
dtype: object

=== HEAD (5 rows) ===
    gene  dCERES  log2FC  ... z_neg_dceres  z_log2fc  sl_score
0   PLK1  -0.239   1.030  ...     1.758307  1.473440  3.231747
1  POLD1  -0.208   1.157  ...     1.374945  1.813738  3.188682
2  FANCN  -0.213   0.977  ...     1.436777  1.331426  2.768204
3  ESPL1  -0.153   1.235  ...     0.694786  2.022739  2.717525
4  SPC24  -0.175   1.130  ...     0.966849  1.741391  2.708240

[5 rows x 11 columns]

=== NULL COUNTS ===
gene            0
dCERES          0
log2FC          0
drug_status     0
depmap_auc      0
pvalue          0
padj 

CRITICAL-1: Print full top-10 data and build Z-score matrix, verify no NaN

In [3]:

import pandas as pd
import numpy as np
from scipy.stats import zscore

df = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')

print("=== FULL DATAFRAME ===")
print(df.to_string())

# Build the 4-metric matrix for clustering
metrics = ['dCERES', 'log2FC', 'depmap_auc', 'padj']
matrix = df[metrics].copy()
matrix.index = df['gene']

print("\n=== RAW METRIC MATRIX (before Z-score) ===")
print(matrix.to_string())

# Apply Z-score normalization column-wise
matrix_z = matrix.apply(zscore)
print("\n=== Z-SCORE NORMALIZED MATRIX ===")
print(matrix_z.to_string())

print("\n=== NaN CHECK ===")
print(matrix_z.isnull().sum())

print("\n=== MATRIX STATS ===")
print(matrix_z.describe())


=== FULL DATAFRAME ===
    gene  dCERES  log2FC drug_status  depmap_auc  pvalue    padj  sl_score_old  z_neg_dceres  z_log2fc  sl_score
0   PLK1  -0.239   1.030    Clinical       0.820  0.0001  0.0093         1.269      1.758307  1.473440  3.231747
1  POLD1  -0.208   1.157       Novel       0.490  0.1222  0.1448         1.365      1.374945  1.813738  3.188682
2  FANCN  -0.213   0.977       Novel       0.523  0.0484  0.1395         1.190      1.436777  1.331426  2.768204
3  ESPL1  -0.153   1.235    Clinical       0.664  0.1296  0.1448         1.388      0.694786  2.022739  2.717525
4  SPC24  -0.175   1.130       Novel       0.740  0.0008  0.0248         1.305      0.966849  1.741391  2.708240
5   CDK1  -0.201   1.000    Clinical       0.800  0.0003  0.0139         1.201      1.288379  1.393055  2.681434
6  CDC45  -0.144   1.260       Novel       0.760  0.0012  0.0279         1.404      0.583487  2.089727  2.673214
7  NDC80  -0.204   0.916       Novel       0.575  0.1210  0.1448         

CRITICAL-1: Regenerate Figure 4 from correct Z-score matrix with ESPL1 fix (MAJOR-2 combined)

In [5]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import zscore
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

# ── Load data ──────────────────────────────────────────────────────────────────
df = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')

# MAJOR-2 fix: ESPL1 drug_status → 'Unverified [SIMULATED]'
df.loc[df['gene'] == 'ESPL1', 'drug_status'] = 'Unverified [SIMULATED]'

# ── Build Z-score matrix ───────────────────────────────────────────────────────
metrics = ['dCERES', 'log2FC', 'depmap_auc', 'padj']
metric_labels = ['dCERES', 'log2FC', 'DepMap AUC', 'padj (BH/FDR)']
matrix = df[metrics].copy()
matrix.index = df['gene']
matrix_z = matrix.apply(zscore)

print("=== CORRECTED Z-SCORE MATRIX (printed to stdout per instructions) ===")
print(matrix_z.round(4).to_string())
print("\nMin:", matrix_z.values.min().round(4), "  Max:", matrix_z.values.max().round(4))
print("Any NaN:", matrix_z.isnull().any().any())

# ── Hierarchical clustering (Ward linkage) ─────────────────────────────────────
dist = pdist(matrix_z.values, metric='euclidean')
Z = linkage(dist, method='ward')

# Get dendrogram order
dend = dendrogram(Z, no_plot=True)
order = dend['leaves']
genes_ordered = matrix_z.index[order].tolist()
matrix_z_ordered = matrix_z.iloc[order]

print("\n=== CLUSTER ORDER ===")
print(genes_ordered)

# ── Drug status color map ──────────────────────────────────────────────────────
drug_colors = {
    'Clinical':              '#E41A1C',
    'Novel':                 '#4DAF4A',
    'Unverified [SIMULATED]':'#FF7F00',
}
df_ordered = df.set_index('gene').loc[genes_ordered]
row_colors = [drug_colors.get(s, '#999999') for s in df_ordered['drug_status']]

# ── Figure layout ──────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(11, 8), facecolor='white')
fig.patch.set_facecolor('white')

# Axes: dendrogram | heatmap | drug-status strip
ax_dend  = fig.add_axes([0.05, 0.12, 0.18, 0.72])
ax_heat  = fig.add_axes([0.25, 0.12, 0.55, 0.72])
ax_strip = fig.add_axes([0.82, 0.12, 0.04, 0.72])

# ── Dendrogram ─────────────────────────────────────────────────────────────────
dendrogram(Z, ax=ax_dend, orientation='left',
           labels=genes_ordered,
           color_threshold=0,
           above_threshold_color='#377EB8',
           leaf_font_size=10)
ax_dend.set_xlabel('Distance', fontsize=8)
ax_dend.spines[['top','right','bottom']].set_visible(False)
ax_dend.tick_params(axis='y', labelsize=10)

# ── Heatmap ────────────────────────────────────────────────────────────────────
cmap = LinearSegmentedColormap.from_list(
    'RdBu_custom', ['#D73027','#FFFFFF','#4575B4'], N=256)

im = ax_heat.imshow(matrix_z_ordered.values,
                    aspect='auto', cmap=cmap,
                    vmin=-2, vmax=2,
                    interpolation='nearest')

ax_heat.set_xticks(range(len(metrics)))
ax_heat.set_xticklabels(metric_labels, fontsize=10, rotation=30, ha='right')
ax_heat.set_yticks(range(len(genes_ordered)))
ax_heat.set_yticklabels(genes_ordered, fontsize=10)
ax_heat.tick_params(axis='both', length=0)

# Annotate cells with Z-score values
for i in range(len(genes_ordered)):
    for j in range(len(metrics)):
        val = matrix_z_ordered.values[i, j]
        color = 'white' if abs(val) > 1.2 else 'black'
        ax_heat.text(j, i, f'{val:.2f}', ha='center', va='center',
                     fontsize=8, color=color, fontweight='bold')

# Colorbar
cbar = fig.colorbar(im, ax=ax_heat, orientation='vertical',
                    fraction=0.03, pad=0.02, shrink=0.8)
cbar.set_label('Z-score', fontsize=9)
cbar.ax.tick_params(labelsize=8)

# ── Drug status strip ──────────────────────────────────────────────────────────
for i, color in enumerate(row_colors):
    ax_strip.add_patch(mpatches.Rectangle((0, i - 0.5), 1, 1,
                                           facecolor=color, edgecolor='white', lw=0.5))
ax_strip.set_xlim(0, 1)
ax_strip.set_ylim(-0.5, len(genes_ordered) - 0.5)
ax_strip.set_yticks([])
ax_strip.set_xticks([])
ax_strip.set_xlabel('Drug\nStatus', fontsize=8, labelpad=4)

# Legend for drug status
legend_patches = [
    mpatches.Patch(color='#E41A1C', label='Clinical'),
    mpatches.Patch(color='#4DAF4A', label='Novel'),
    mpatches.Patch(color='#FF7F00', label='Unverified [SIMULATED]'),
]
ax_strip.legend(handles=legend_patches, loc='lower right',
                bbox_to_anchor=(4.5, -0.18), fontsize=8,
                frameon=True, framealpha=0.9, ncol=3)

# ── Title & caption ────────────────────────────────────────────────────────────
fig.suptitle(
    'Figure 4. Hierarchical Clustering of Top-10 siRNA Candidates\n'
    '(Z-score normalized; Ward linkage; corrected from prior NaN-substitution run)',
    fontsize=11, fontweight='bold', y=0.97, color='black'
)

fig.text(0.5, 0.02,
    'SIMULATED DATA — for methodological demonstration only. '
    'Z-score normalization applied to 4 metrics × 10 genes. '
    'A prior run produced a degenerate all-zero matrix due to a column-name mismatch\n'
    '(NaN→0 substitution); this has been corrected. '
    'ESPL1 drug status is unverified in real data.',
    ha='center', fontsize=7.5, color='#555555', style='italic',
    wrap=True)

plt.savefig('/mnt/results/figures/Figure4.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.savefig('/mnt/results/figures/Figure4.svg', bbox_inches='tight',
            facecolor='white')
plt.close()
print("\nFigure 4 saved.")


=== CORRECTED Z-SCORE MATRIX (printed to stdout per instructions) ===
       dCERES  log2FC  depmap_auc    padj
gene                                     
PLK1  -1.6827 -0.3822      1.2603 -1.2752
POLD1 -0.6815  0.7234     -1.3972  0.9957
FANCN -0.8429 -0.8436     -1.1315  0.9068
ESPL1  1.0949  1.4025      0.0040  0.9957
SPC24  0.3843  0.4884      0.6161 -1.0154
CDK1  -0.4554 -0.6433      1.0993 -1.1981
CDC45  1.3855  1.6201      0.7771 -0.9635
NDC80 -0.5523 -1.3746     -0.7127  0.9957
ATM   -0.1324 -1.2005     -1.3409  0.9957
CENPN  1.4824  0.2098      0.8255 -0.4372

Min: -1.6827   Max: 1.6201
Any NaN: False

=== CLUSTER ORDER ===
['POLD1', 'ATM', 'FANCN', 'NDC80', 'PLK1', 'CDK1', 'ESPL1', 'CDC45', 'SPC24', 'CENPN']

Figure 4 saved.


CRITICAL-1: Regenerate Figure 4 with fixed layout — gene names, legend, spacing

In [7]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import zscore
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

# ── Load & fix ESPL1 ───────────────────────────────────────────────────────────
df = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')
df.loc[df['gene'] == 'ESPL1', 'drug_status'] = 'Unverified [SIMULATED]'

# ── Z-score matrix ─────────────────────────────────────────────────────────────
metrics       = ['dCERES', 'log2FC', 'depmap_auc', 'padj']
metric_labels = ['dCERES', 'log2FC', 'DepMap AUC', 'padj\n(BH/FDR)']
matrix        = df[metrics].copy()
matrix.index  = df['gene']
matrix_z      = matrix.apply(zscore)

# ── Clustering ─────────────────────────────────────────────────────────────────
dist  = pdist(matrix_z.values, metric='euclidean')
Z_lnk = linkage(dist, method='ward')
dend  = dendrogram(Z_lnk, no_plot=True)
order = dend['leaves']
genes_ordered    = matrix_z.index[order].tolist()
matrix_z_ordered = matrix_z.iloc[order]
df_ordered       = df.set_index('gene').loc[genes_ordered]

# ── Drug status colors ─────────────────────────────────────────────────────────
drug_colors = {
    'Clinical':               '#E41A1C',
    'Novel':                  '#4DAF4A',
    'Unverified [SIMULATED]': '#FF7F00',
}
row_colors = [drug_colors.get(s, '#999999') for s in df_ordered['drug_status']]

# ── Figure: wider canvas, generous margins ─────────────────────────────────────
fig = plt.figure(figsize=(13, 8), facecolor='white')

# [left, bottom, width, height]
ax_dend  = fig.add_axes([0.02, 0.18, 0.14, 0.68])   # dendrogram
ax_heat  = fig.add_axes([0.28, 0.18, 0.50, 0.68])   # heatmap
ax_strip = fig.add_axes([0.80, 0.18, 0.03, 0.68])   # drug strip
ax_cbar  = fig.add_axes([0.85, 0.35, 0.02, 0.34])   # colorbar

# ── Dendrogram ─────────────────────────────────────────────────────────────────
dendrogram(Z_lnk, ax=ax_dend, orientation='left',
           labels=genes_ordered,
           color_threshold=0,
           above_threshold_color='#377EB8',
           leaf_font_size=11)
ax_dend.set_xlabel('Distance', fontsize=8)
ax_dend.spines[['top','right','bottom']].set_visible(False)
ax_dend.tick_params(axis='y', labelsize=11)
# Remove y-tick labels from dendrogram (gene names shown on heatmap)
ax_dend.set_yticklabels([])

# ── Heatmap ────────────────────────────────────────────────────────────────────
cmap = LinearSegmentedColormap.from_list(
    'RdBu_custom', ['#D73027', '#FFFFFF', '#4575B4'], N=256)

im = ax_heat.imshow(matrix_z_ordered.values,
                    aspect='auto', cmap=cmap,
                    vmin=-2, vmax=2,
                    interpolation='nearest')

ax_heat.set_xticks(range(len(metrics)))
ax_heat.set_xticklabels(metric_labels, fontsize=10, rotation=0, ha='center')
ax_heat.set_yticks(range(len(genes_ordered)))
ax_heat.set_yticklabels(genes_ordered, fontsize=11, fontweight='bold')
ax_heat.tick_params(axis='both', length=0)
ax_heat.yaxis.set_tick_params(pad=6)

# Annotate cells
for i in range(len(genes_ordered)):
    for j in range(len(metrics)):
        val = matrix_z_ordered.values[i, j]
        # For near-zero, use black; for strong values use white
        color = 'white' if abs(val) > 1.3 else 'black'
        ax_heat.text(j, i, f'{val:.2f}', ha='center', va='center',
                     fontsize=9, color=color, fontweight='bold')

# ── Colorbar ───────────────────────────────────────────────────────────────────
cbar = fig.colorbar(im, cax=ax_cbar, orientation='vertical')
cbar.set_label('Z-score', fontsize=9, labelpad=6)
cbar.ax.tick_params(labelsize=8)

# ── Drug status strip ──────────────────────────────────────────────────────────
for i, color in enumerate(row_colors):
    ax_strip.add_patch(mpatches.Rectangle(
        (0, i - 0.5), 1, 1,
        facecolor=color, edgecolor='white', lw=0.8))
ax_strip.set_xlim(0, 1)
ax_strip.set_ylim(-0.5, len(genes_ordered) - 0.5)
ax_strip.set_yticks([])
ax_strip.set_xticks([])
ax_strip.set_title('Drug\nStatus', fontsize=8, pad=4)

# Legend — placed directly below the strip
legend_patches = [
    mpatches.Patch(color='#E41A1C', label='Clinical'),
    mpatches.Patch(color='#4DAF4A', label='Novel'),
    mpatches.Patch(color='#FF7F00', label='Unverified\n[SIMULATED]'),
]
fig.legend(handles=legend_patches,
           loc='lower right',
           bbox_to_anchor=(0.99, 0.01),
           fontsize=8,
           frameon=True, framealpha=0.9,
           title='Drug Status', title_fontsize=8,
           ncol=1)

# ── Title ──────────────────────────────────────────────────────────────────────
fig.suptitle(
    'Figure 4. Hierarchical Clustering of Top-10 siRNA Candidates\n'
    '(Z-score normalized; Ward linkage; 4 metrics × 10 genes)',
    fontsize=12, fontweight='bold', y=0.97, color='black'
)

# ── Footer caption ─────────────────────────────────────────────────────────────
fig.text(
    0.5, 0.01,
    'SIMULATED DATA — for methodological demonstration only. '
    'A prior run produced a degenerate all-zero matrix due to a column-name mismatch '
    '(NaN→0 substitution); this has been corrected in the current figure. '
    'ESPL1 drug status is unverified in real data.',
    ha='center', fontsize=7.5, color='#555555', style='italic',
    wrap=True
)

plt.savefig('/mnt/results/figures/Figure4.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.savefig('/mnt/results/figures/Figure4.svg', bbox_inches='tight',
            facecolor='white')
plt.close()
print("Figure 4 regenerated.")


Figure 4 regenerated.


CRITICAL-1 final: Fix footer truncation and legend overlap in Figure 4

In [9]:

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import zscore
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

# ── Load & fix ESPL1 ───────────────────────────────────────────────────────────
df = pd.read_csv('/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv')
df.loc[df['gene'] == 'ESPL1', 'drug_status'] = 'Unverified [SIMULATED]'

# ── Z-score matrix ─────────────────────────────────────────────────────────────
metrics       = ['dCERES', 'log2FC', 'depmap_auc', 'padj']
metric_labels = ['dCERES', 'log2FC', 'DepMap AUC', 'padj\n(BH/FDR)']
matrix        = df[metrics].copy()
matrix.index  = df['gene']
matrix_z      = matrix.apply(zscore)

# ── Clustering ─────────────────────────────────────────────────────────────────
dist  = pdist(matrix_z.values, metric='euclidean')
Z_lnk = linkage(dist, method='ward')
dend  = dendrogram(Z_lnk, no_plot=True)
order = dend['leaves']
genes_ordered    = matrix_z.index[order].tolist()
matrix_z_ordered = matrix_z.iloc[order]
df_ordered       = df.set_index('gene').loc[genes_ordered]

# ── Drug status colors ─────────────────────────────────────────────────────────
drug_colors = {
    'Clinical':               '#E41A1C',
    'Novel':                  '#4DAF4A',
    'Unverified [SIMULATED]': '#FF7F00',
}
row_colors = [drug_colors.get(s, '#999999') for s in df_ordered['drug_status']]

# ── Figure ─────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 9), facecolor='white')

# Axes positions: [left, bottom, width, height]
ax_dend  = fig.add_axes([0.02, 0.20, 0.13, 0.66])
ax_heat  = fig.add_axes([0.27, 0.20, 0.48, 0.66])
ax_strip = fig.add_axes([0.77, 0.20, 0.03, 0.66])
ax_cbar  = fig.add_axes([0.83, 0.38, 0.02, 0.30])

# ── Dendrogram ─────────────────────────────────────────────────────────────────
dendrogram(Z_lnk, ax=ax_dend, orientation='left',
           labels=genes_ordered,
           color_threshold=0,
           above_threshold_color='#377EB8',
           leaf_font_size=11)
ax_dend.set_xlabel('Distance', fontsize=8)
ax_dend.spines[['top','right','bottom']].set_visible(False)
ax_dend.set_yticklabels([])

# ── Heatmap ────────────────────────────────────────────────────────────────────
cmap = LinearSegmentedColormap.from_list(
    'RdBu_custom', ['#D73027', '#FFFFFF', '#4575B4'], N=256)

im = ax_heat.imshow(matrix_z_ordered.values,
                    aspect='auto', cmap=cmap,
                    vmin=-2, vmax=2,
                    interpolation='nearest')

ax_heat.set_xticks(range(len(metrics)))
ax_heat.set_xticklabels(metric_labels, fontsize=10, rotation=0, ha='center')
ax_heat.set_yticks(range(len(genes_ordered)))
ax_heat.set_yticklabels(genes_ordered, fontsize=11, fontweight='bold')
ax_heat.tick_params(axis='both', length=0)
ax_heat.yaxis.set_tick_params(pad=6)

# Cell annotations
for i in range(len(genes_ordered)):
    for j in range(len(metrics)):
        val = matrix_z_ordered.values[i, j]
        color = 'white' if abs(val) > 1.3 else 'black'
        ax_heat.text(j, i, f'{val:.2f}', ha='center', va='center',
                     fontsize=9, color=color, fontweight='bold')

# ── Colorbar ───────────────────────────────────────────────────────────────────
cbar = fig.colorbar(im, cax=ax_cbar, orientation='vertical')
cbar.set_label('Z-score', fontsize=9, labelpad=8)
cbar.ax.tick_params(labelsize=8)

# ── Drug status strip ──────────────────────────────────────────────────────────
for i, color in enumerate(row_colors):
    ax_strip.add_patch(mpatches.Rectangle(
        (0, i - 0.5), 1, 1,
        facecolor=color, edgecolor='white', lw=0.8))
ax_strip.set_xlim(0, 1)
ax_strip.set_ylim(-0.5, len(genes_ordered) - 0.5)
ax_strip.set_yticks([])
ax_strip.set_xticks([])
ax_strip.set_title('Drug\nStatus', fontsize=8, pad=4)

# Legend — placed below colorbar, well clear of footer
legend_patches = [
    mpatches.Patch(color='#E41A1C', label='Clinical'),
    mpatches.Patch(color='#4DAF4A', label='Novel'),
    mpatches.Patch(color='#FF7F00', label='Unverified\n[SIMULATED]'),
]
fig.legend(handles=legend_patches,
           loc='lower right',
           bbox_to_anchor=(0.99, 0.10),   # raised above footer
           fontsize=8,
           frameon=True, framealpha=0.9,
           title='Drug Status', title_fontsize=8,
           ncol=1)

# ── Title ──────────────────────────────────────────────────────────────────────
fig.suptitle(
    'Figure 4. Hierarchical Clustering of Top-10 siRNA Candidates\n'
    '(Z-score normalized; Ward linkage; 4 metrics × 10 genes)',
    fontsize=12, fontweight='bold', y=0.98, color='black'
)

# ── Footer — two lines, constrained width ──────────────────────────────────────
fig.text(
    0.5, 0.055,
    'SIMULATED DATA — for methodological demonstration only. '
    'Z-score normalization applied to 4 metrics × 10 genes.',
    ha='center', fontsize=7.5, color='#555555', style='italic'
)
fig.text(
    0.5, 0.025,
    'A prior run produced a degenerate all-zero matrix due to a column-name mismatch '
    '(NaN→0 substitution); this has been corrected. ESPL1 drug status is unverified in real data.',
    ha='center', fontsize=7.5, color='#555555', style='italic'
)

plt.savefig('/mnt/results/figures/Figure4.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.savefig('/mnt/results/figures/Figure4.svg', bbox_inches='tight',
            facecolor='white')
plt.close()
print("Figure 4 final version saved.")


Figure 4 final version saved.


MAJOR-2 + MINOR-1: Fix ESPL1 drug_status and rename depmap_auc→ceres_selectivity in CSVs

In [11]:

import pandas as pd

# ── 1. Fix SIMULATED_top10_siRNA_candidates.csv ────────────────────────────────
top10_path = '/mnt/results/data/SIMULATED_top10_siRNA_candidates.csv'
df10 = pd.read_csv(top10_path)

print("BEFORE — top10 columns:", df10.columns.tolist())
print("BEFORE — ESPL1 drug_status:", df10.loc[df10['gene']=='ESPL1', 'drug_status'].values)

# MAJOR-2: fix ESPL1 drug status
df10.loc[df10['gene'] == 'ESPL1', 'drug_status'] = 'Unverified [SIMULATED]'

# MINOR-1: rename depmap_auc → ceres_selectivity
df10 = df10.rename(columns={'depmap_auc': 'ceres_selectivity'})

print("\nAFTER — top10 columns:", df10.columns.tolist())
print("AFTER — ESPL1 drug_status:", df10.loc[df10['gene']=='ESPL1', 'drug_status'].values)

df10.to_csv(top10_path, index=False)
print("top10 CSV saved.")

# ── 2. Fix SIMULATED_DepMap_93_SL_candidates.csv ───────────────────────────────
dep_path = '/mnt/results/data/SIMULATED_DepMap_93_SL_candidates.csv'
df93 = pd.read_csv(dep_path)

print("\nBEFORE — dep93 columns:", df93.columns.tolist())

if 'depmap_auc' in df93.columns:
    df93 = df93.rename(columns={'depmap_auc': 'ceres_selectivity'})
    print("Renamed depmap_auc → ceres_selectivity in dep93")
else:
    print("depmap_auc not found in dep93 — checking existing columns")

# Also fix ESPL1 if present
if 'drug_status' in df93.columns and 'gene' in df93.columns:
    mask = df93['gene'] == 'ESPL1'
    if mask.any():
        df93.loc[mask, 'drug_status'] = 'Unverified [SIMULATED]'
        print("ESPL1 drug_status fixed in dep93")

print("AFTER — dep93 columns:", df93.columns.tolist())
df93.to_csv(dep_path, index=False)
print("dep93 CSV saved.")

# ── 3. Verify ──────────────────────────────────────────────────────────────────
print("\n=== VERIFICATION ===")
v10 = pd.read_csv(top10_path)
print("top10 columns:", v10.columns.tolist())
print("top10 ESPL1 row:")
print(v10[v10['gene']=='ESPL1'][['gene','drug_status','ceres_selectivity']].to_string())

v93 = pd.read_csv(dep_path)
print("\ndep93 columns:", v93.columns.tolist())
if 'gene' in v93.columns and 'drug_status' in v93.columns:
    espl1_93 = v93[v93['gene']=='ESPL1']
    if not espl1_93.empty:
        print("dep93 ESPL1 drug_status:", espl1_93['drug_status'].values)


BEFORE — top10 columns: ['gene', 'dCERES', 'log2FC', 'drug_status', 'depmap_auc', 'pvalue', 'padj', 'sl_score_old', 'z_neg_dceres', 'z_log2fc', 'sl_score']
BEFORE — ESPL1 drug_status: ['Clinical']

AFTER — top10 columns: ['gene', 'dCERES', 'log2FC', 'drug_status', 'ceres_selectivity', 'pvalue', 'padj', 'sl_score_old', 'z_neg_dceres', 'z_log2fc', 'sl_score']
AFTER — ESPL1 drug_status: ['Unverified [SIMULATED]']
top10 CSV saved.

BEFORE — dep93 columns: ['gene', 'dCERES', 'log2FC', 'drug_status', 'depmap_auc', 'pvalue', 'padj', 'venn_category']
Renamed depmap_auc → ceres_selectivity in dep93
ESPL1 drug_status fixed in dep93
AFTER — dep93 columns: ['gene', 'dCERES', 'log2FC', 'drug_status', 'ceres_selectivity', 'pvalue', 'padj', 'venn_category']
dep93 CSV saved.

=== VERIFICATION ===
top10 columns: ['gene', 'dCERES', 'log2FC', 'drug_status', 'ceres_selectivity', 'pvalue', 'padj', 'sl_score_old', 'z_neg_dceres', 'z_log2fc', 'sl_score']
top10 ESPL1 row:
    gene             drug_status  cer